# 00 — Setup and pre-registration (run once, CPU is fine)

v3 of *Diversity Is Not Enough* (TMLR #10271). This notebook:

1. writes the analysis code to `MyDrive/TMLR_v3/code/`,
2. checks that your datasets and the CTransPath checkpoint are where the code expects them,
3. checks Hugging Face access for the gated pathology encoders,
4. runs the test suite (a hard gate),
5. **registers** the analysis (`prereg.json`).

**Run order:** `00` → `01` (pathology, GPU) → `02` (vision, GPU) → `03` (text, GPU) → `04` (analysis, CPU or GPU).
01, 02 and 03 are independent and resumable; re-run any of them after a disconnect and it continues where it stopped.

**Before you start:** for the gated encoders, accept the licence on each model page while logged in to Hugging Face
(UNI, UNI2-h, Virchow2, H-optimus-0, Prov-GigaPath, Hibou-B), then add your token as a Colab secret named
`HF_TOKEN` (key icon in the left sidebar → enable notebook access). Without it those six are skipped and the
pathology pools have 6 encoders instead of 12.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
PROJECT = "/content/drive/MyDrive/TMLR_v3"
for d in ("code", "tests", "features", "manifests", "images224", "results", "logs"):
    os.makedirs(os.path.join(PROJECT, d), exist_ok=True)
print("project folder:", PROJECT)

## 1. Write the code to Drive
Each cell below writes one module. They were tested before delivery; do not edit them after step 5 (registration).

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/config.py
"""Single source of truth for paths, datasets, encoders and the pre-registered
analysis parameters of v3.

Everything here is hashed into prereg.json by notebook 00. Notebook 04 refuses
to analyse if this file (or any analysis module) changed after registration,
unless you record a deviation explicitly. Edit the PATHS block if your Drive
layout differs; do not edit PREREG after registering.
"""

from __future__ import annotations

import os

# ---------------------------------------------------------------------------
# PATHS  (taken from your existing Drive layout; edit if anything moved)
# ---------------------------------------------------------------------------
DRIVE = os.environ.get("V3_DRIVE") or next(
    (d for d in ("/content/drive/MyDrive", "/content/drive/My Drive") if os.path.isdir(d)),
    "/content/drive/MyDrive")
HET = os.path.join(DRIVE, "HetFuseKD")

PROJECT = os.path.join(DRIVE, "TMLR_v3")          # everything v3 writes lives here
CODE_DIR = os.path.join(PROJECT, "code")
FEAT_DIR = os.path.join(PROJECT, "features")      # features/<dataset>/<encoder>.npy
MANIFEST_DIR = os.path.join(PROJECT, "manifests")  # manifests/<dataset>.csv + .json
IMG_DRIVE_DIR = os.path.join(PROJECT, "images224")  # persisted image caches (Drive-sourced sets only)
RESULTS_DIR = os.path.join(PROJECT, "results")
LOG_DIR = os.path.join(PROJECT, "logs")
PREREG_PATH = os.path.join(PROJECT, "prereg.json")

LOCAL = os.environ.get("V3_LOCAL", "/content/v3_local")  # fast scratch; lost on disconnect
LOCAL_IMG = os.path.join(LOCAL, "images224")
LOCAL_DL = os.path.join(LOCAL, "downloads")
LOCAL_SHARDS = os.path.join(LOCAL, "shards")

# your datasets already on Drive
BRACS_DIR = os.path.join(HET, "data", "BRACS_RoI", "latest_version")   # train/ val/ test/
SICAP_DIR = os.path.join(DRIVE, "SICAPv2")                             # images/ partition/Test/
MHIST_DIR = os.path.join(DRIVE, "MHIST")                               # images/ + annotations csv
MHIST_CSV_CANDIDATES = ("annotations (1).csv", "annotations.csv")
CTRANSPATH_CKPT = os.path.join(HET, "model_weights", "ctranspath.pth")

# v2 caches, used ONLY by the consistency check in notebook 01
V2_CACHE = {
    "bracs":   os.path.join(HET, "cached_features_bracs"),
    "sicapv2": os.path.join(HET, "cached_features_sicapv2"),
    "mhist":   os.path.join(HET, "cached_features_mhist"),
}
V2_CTRANSPATH_FIXED = {ds: os.path.join(HET, "expanded", ds, "ctranspath_fixed")
                       for ds in ("bracs", "sicapv2", "mhist")}
V2_NAME = {"uni": "uni", "phikon_v2": "phikon_v2", "ctranspath": "ctranspath",
           "resnet50_in1k": "resnet50_imagenet"}

# public downloads (all verified reachable on 2026-10-21)
ZENODO_NCT = "https://zenodo.org/api/records/1214456/files/{}/content"
ZENODO_PCAM = "https://zenodo.org/api/records/2546921/files/{}/content"

# ---------------------------------------------------------------------------
# DATASETS
#   cap      : max rows kept (stratified, seeded); None = all
#   persist  : also save the 224px uint8 image cache to Drive (only worth it for
#              sets that are slow to re-read, i.e. those stored on Drive)
# ---------------------------------------------------------------------------
IMAGE_SIZE = 224
SEED = 0

PATHOLOGY_DATASETS = {
    "bracs":      dict(cap=None,   persist=True,  note="train+val RoIs (3,969); grouped by WSI id"),
    "sicapv2":    dict(cap=None,   persist=True,  note="partition/Test Train+Test (12,081); grouped by biopsy"),
    "mhist":      dict(cap=None,   persist=True,  note="all 3,152; no patient ids released -> ungrouped"),
    "crc_val_7k": dict(cap=None,   persist=False, note="CRC-VAL-HE-7K, Zenodo 1214456; ungrouped"),
    "nct_crc_100k": dict(cap=20000, persist=False, note="NCT-CRC-HE-100K (11.7 GB download); ungrouped"),
    "pcam":       dict(cap=20000,  persist=False, note="PCam test split (32,768 -> 20k); grouped by WSI"),
}

VISION_DATASETS = {
    "dtd":        dict(cap=None,  persist=False, note="torchvision split=train (1,880), as v2"),
    "aircraft":   dict(cap=None,  persist=False, note="torchvision split=train (3,334), as v2"),
    "eurosat":    dict(cap=20000, persist=False, note="27,000 -> 20k (v2 used all)"),
    "pets":       dict(cap=None,  persist=False, note="OxfordIIITPet trainval (3,680)"),
    "food101":    dict(cap=20000, persist=False, note="train 75,750 -> 20k"),
    "cifar100":   dict(cap=20000, persist=False, note="train 50,000 -> 20k; 32px upsampled"),
    "gtsrb":      dict(cap=20000, persist=False, note="train 26,640 -> 20k"),
    "country211": dict(cap=20000, persist=False, note="train 31,650 -> 20k"),
}

TEXT_DATASETS = {
    # name: (hf_id, config, split, text_col, label_col, cap)
    "emotion":    ("SetFit/emotion", None, "train", "text", "label", 20000),
    "ag_news":    ("fancyzhx/ag_news", None, "train", "text", "label", 20000),
    "banking77":  ("mteb/banking77", None, "train", "text", "label", 20000),
    "trec_coarse": ("SetFit/TREC-QC", None, "train", "text", "label_coarse", 20000),
    "newsgroups20": ("SetFit/20_newsgroups", None, "train", "text", "label", 20000),
    "sst5":       ("SetFit/sst5", None, "train", "text", "label", 20000),
}

# ---------------------------------------------------------------------------
# ENCODERS
#   kind   : timm | hf | ctranspath | st (sentence-transformers)
#   pool   : default (model's own pooled output) | cls+meanpatch (Virchow-style)
#   gated  : needs an accepted licence on Hugging Face + HF_TOKEN in Colab secrets
# Every repo id below was checked to exist on 2026-10-21.
# ---------------------------------------------------------------------------
PATHOLOGY_ENCODERS = {
    # the four v2 encoders
    "uni":           dict(kind="timm", id="hf-hub:MahmoodLab/uni", gated=True,
                          kw=dict(init_values=1e-5, dynamic_img_size=True)),
    "phikon_v2":     dict(kind="hf", id="owkin/phikon-v2", pool="cls"),
    "ctranspath":    dict(kind="ctranspath"),
    "resnet50_in1k": dict(kind="timm", id="resnet50.tv_in1k"),
    # new public pathology encoders
    "phikon":        dict(kind="hf", id="owkin/phikon", pool="cls"),
    "uni2_h":        dict(kind="timm", id="hf-hub:MahmoodLab/UNI2-h", gated=True, kw="UNI2_H"),
    "virchow2":      dict(kind="timm", id="hf-hub:paige-ai/Virchow2", gated=True, kw="SWIGLU",
                          pool="cls+meanpatch"),
    "h_optimus_0":   dict(kind="timm", id="hf-hub:bioptimus/H-optimus-0", gated=True,
                          kw=dict(init_values=1e-5, dynamic_img_size=False)),
    "prov_gigapath": dict(kind="timm", id="hf-hub:prov-gigapath/prov-gigapath", gated=True),
    "hibou_b":       dict(kind="hf", id="histai/hibou-b", gated=True, pool="pooler",
                          trust_remote_code=True),
    "lunit_vits16":  dict(kind="timm", id="hf-hub:1aurent/vit_small_patch16_224.lunit_dino"),
    "kaiko_vitb16":  dict(kind="timm",
                          id="hf-hub:1aurent/vit_base_patch16_224.kaiko_ai_towards_large_pathology_fms",
                          kw=dict(dynamic_img_size=True), mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
}

VISION_ENCODERS = {
    # the six v2 encoders (same timm tags as the v2 w3 cache)
    "dinov2_l":     dict(kind="timm", id="vit_large_patch14_dinov2.lvd142m"),
    "eva02_clip":   dict(kind="timm", id="eva02_large_patch14_clip_224.merged2b"),
    "swin_l":       dict(kind="timm", id="swin_large_patch4_window7_224.ms_in22k"),
    "convnext_l":   dict(kind="timm", id="convnext_large.fb_in22k"),
    "resnet50":     dict(kind="timm", id="resnet50.a1_in1k"),
    "vit_mae":      dict(kind="timm", id="vit_base_patch16_224.mae"),
    # new
    "clip_b32":     dict(kind="timm", id="vit_base_patch32_clip_224.openai"),
    "siglip_b16":   dict(kind="timm", id="vit_base_patch16_siglip_224.webli"),
    "dinov2_b":     dict(kind="timm", id="vit_base_patch14_dinov2.lvd142m"),
    "openclip_h14": dict(kind="timm", id="vit_huge_patch14_clip_224.laion2b"),
}

TEXT_ENCODERS = {
    "minilm_l6":   dict(kind="st", id="sentence-transformers/all-MiniLM-L6-v2"),
    "mpnet_base":  dict(kind="st", id="sentence-transformers/all-mpnet-base-v2"),
    "distilroberta": dict(kind="st", id="sentence-transformers/all-distilroberta-v1"),
    "bge_small":   dict(kind="st", id="BAAI/bge-small-en-v1.5"),
    "bge_base":    dict(kind="st", id="BAAI/bge-base-en-v1.5"),
    "gte_base":    dict(kind="st", id="thenlper/gte-base"),
    "e5_base":     dict(kind="st", id="intfloat/e5-base-v2", prefix="query: "),
    "minilm_l3":   dict(kind="st", id="sentence-transformers/paraphrase-MiniLM-L3-v2"),
    "glove_avg":   dict(kind="st", id="sentence-transformers/average_word_embeddings_glove.6B.300d"),
}

DOMAINS = {
    "pathology": (PATHOLOGY_DATASETS, PATHOLOGY_ENCODERS),
    "vision": (VISION_DATASETS, VISION_ENCODERS),
    "text": (TEXT_DATASETS, TEXT_ENCODERS),
}

FEATURE_DTYPE = "float32"    # float16 halves Drive use but adds ~0.01 CKA error (v2 App. B.4)
EXTRACT_BATCH = 64
SHARD_ROWS = 2048            # features are flushed to Drive every SHARD_ROWS rows

# ---------------------------------------------------------------------------
# PRE-REGISTERED ANALYSIS PARAMETERS  (do not edit after notebook 00 registers)
# ---------------------------------------------------------------------------
PREREG = dict(
    unit_of_inference="pool (= one dataset with all available encoders of its domain)",
    k_folds=5,
    seed=SEED,
    grouped_folds_where_groups_exist=True,
    probe="StandardScaler (train fold) + L2 logistic regression, C=1.0",
    probe_backend="torch",        # exact optimum; sklearn default tol stops early (tests/)
    anchor_rule="highest individual held-out accuracy",
    base_rule="anchor + median-CKA member (CKA ordering), fixed before pairing",
    comp_tol_pp=1.5,
    cka_min_gap=0.15,
    cka_sd_floor=3.0,
    n_cap_cka=10000,
    primary_fusion_rule="mean_logits",
    secondary_fusion_rules=["mean_probs", "class_conditional"],
    primary_estimand="A: cluster-robust (by pool) mean of D over the ONE primary matched "
                     "pair per pool (max CKA gap), t(G-1) reference",
    equivalence_margins_pp=[1.0, 0.5, 0.25],
    dose_response="D on CKA gap over all matched pairs, cluster-robust by pool; "
                  "interpret slope only if MDE(80%) <= 0.5 pp per unit CKA",
    regret_criteria=["cka_diversity", "disagreement_with_base", "competence", "oracle_headroom"],
    regret_note="only cka_diversity is label-free; disagreement uses probe predictions",
    mechanism="utility ~ disagreement + acc ; utility ~ cka + acc ; joint; cluster-robust",
    analyses=["all pools", "pathology", "vision", "text", "image pools (pathology+vision)"],
    min_encoders_per_pool=5,
)

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/cka.py
"""Linear Centered Kernel Alignment (Kornblith et al., 2019).

Memory/scaling notes for Colab Pro (2 vCPU, ~13-25 GB RAM):
  The d x d Gram formulation below costs O(n * di * dj) and O(di * dj) memory,
  which is independent of n for storage. For a 20k-sample pool of 24 encoders
  (276 pairs) this is seconds, not hours.

  The paper's released subsets reproduce full-sample CKA to within 0.013
  (half-precision quantisation) with a subsampling SD of 0.003-0.006. We
  therefore compute the full pairwise matrix on a fixed n_cap subsample and
  report the subsampling SD explicitly, rather than pretending to exactness.
"""

from __future__ import annotations

import numpy as np


def center(F: np.ndarray) -> np.ndarray:
    """Column-center a feature matrix (n x d)."""
    F = np.asarray(F, dtype=np.float64)
    return F - F.mean(axis=0, keepdims=True)


def linear_cka(Fi: np.ndarray, Fj: np.ndarray) -> float:
    """Linear CKA between two feature matrices on the *same* inputs, same order.

        CKA = ||Fj^T Fi||_F^2 / (||Fi^T Fi||_F * ||Fj^T Fj||_F)

    Both matrices are column-centred first. Returns a float in [0, 1].
    """
    Fi = center(Fi)
    Fj = center(Fj)
    if Fi.shape[0] != Fj.shape[0]:
        raise ValueError(
            f"CKA requires the same number of inputs, got {Fi.shape[0]} and {Fj.shape[0]}"
        )
    G = Fj.T @ Fi                      # (dj, di) -- the only n-dependent cost
    num = float(np.sum(G * G))         # ||G||_F^2
    di = float(np.sum((Fi.T @ Fi) ** 2))
    dj = float(np.sum((Fj.T @ Fj) ** 2))
    if di <= 0.0 or dj <= 0.0:
        raise ValueError("degenerate feature matrix (zero self-alignment)")
    return num / (np.sqrt(di) * np.sqrt(dj))


def pairwise_cka_matrix(feats: dict[str, np.ndarray], n_cap: int | None = None,
                        seed: int = 0) -> tuple[np.ndarray, list[str]]:
    """Full pairwise CKA matrix over a dict {name: features}.

    All feature matrices must be row-aligned (same inputs, same order). If
    n_cap is given, a single fixed subsample of rows is used for every pair so
    that all entries are mutually comparable.
    """
    names = sorted(feats)
    mats = {k: np.asarray(feats[k], dtype=np.float64) for k in names}
    ns = {m.shape[0] for m in mats.values()}
    if len(ns) != 1:
        raise ValueError(f"feature matrices are not row-aligned: row counts {ns}")
    n = ns.pop()

    if n_cap is not None and n_cap < n:
        rng = np.random.default_rng(seed)
        idx = np.sort(rng.choice(n, size=n_cap, replace=False))
        mats = {k: v[idx] for k, v in mats.items()}

    K = len(names)
    C = np.eye(K)
    for i in range(K):
        for j in range(i + 1, K):
            v = linear_cka(mats[names[i]], mats[names[j]])
            C[i, j] = C[j, i] = v
    return C, names


def cka_subsampling_sd(Fi: np.ndarray, Fj: np.ndarray, n_draws: int = 50,
                       n_sub: int = 2000, seed: int = 0) -> float:
    """SD of the CKA estimate across independent subsamples of n_sub inputs.

    This is the uncertainty the paper reports (0.003-0.006). Report it next to
    every CKA gap you interpret: a gap that does not exceed ~3 SD carries no
    information, which is exactly the failure mode of the MHIST comparison
    (gap 0.012) in the rejected version.
    """
    Fi = np.asarray(Fi); Fj = np.asarray(Fj)
    n = Fi.shape[0]
    n_sub = min(n_sub, n)
    rng = np.random.default_rng(seed)
    vals = np.empty(n_draws)
    for d in range(n_draws):
        idx = rng.choice(n, size=n_sub, replace=False)
        vals[d] = linear_cka(Fi[idx], Fj[idx])
    return float(vals.std(ddof=1))

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/folds.py
"""Per-fold logit protocol.

The rejected version got this right and it should not change: a probe fit on all
data and scored on the same data inflates apparent teacher accuracy by up to 10
points, and that inflation propagates into every oracle bound, unique-contribution
count and fusion gain. Everything downstream depends on these being held-out.

New in this version: optional *group* awareness. If the dataset has a group
variable (patient, slide, institution), folds MUST be grouped by it. The
SICAPv2 patient-level result in Appendix K showed that patch-level folds leak:
patches from one biopsy appear in both train and validation, which inflates
in-distribution numbers and hides the shift failure. Group from the start.
"""

from __future__ import annotations

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler


def make_folds(y: np.ndarray, groups: np.ndarray | None = None, k: int = 5,
               seed: int = 0) -> list[np.ndarray]:
    """Return a list of k boolean holdout masks over the inputs."""
    y = np.asarray(y)
    n = y.shape[0]
    masks: list[np.ndarray] = []
    if groups is not None:
        groups = np.asarray(groups)
        splitter = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
        iterator = splitter.split(np.zeros(n), y, groups)
    else:
        # stratify when feasible, else plain KFold (e.g. very few samples/class)
        _, counts = np.unique(y, return_counts=True)
        if counts.min() >= k:
            splitter = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
        else:
            splitter = KFold(n_splits=k, shuffle=True, random_state=seed)
        iterator = splitter.split(np.zeros(n), y)
    for _, va in iterator:
        m = np.zeros(n, dtype=bool)
        m[va] = True
        masks.append(m)
    # every input is held out exactly once
    assert np.sum(np.stack(masks).astype(int), axis=0).min() == 1
    if groups is not None:
        for m in masks:
            assert len(np.intersect1d(groups[m], groups[~m])) == 0, "group leakage"
    return masks


def _torch_probe(Xtr, ytr, Xva, n_classes, C=1.0, max_iter=500, device=None):
    """Same objective as sklearn's LogisticRegression(C, lbfgs) solved with
    torch L-BFGS, so large pools (20k x 2560 x 211 classes) fit in seconds on a
    GPU instead of minutes on CPU.

    sklearn minimises  C * sum_i CE_i + 0.5 * ||W||^2  (bias unpenalised).
    Dividing by C*n gives  mean CE + ||W||^2 / (2 C n), which is what we use.
    Binary problems use a single weight vector, exactly as sklearn does, because
    a two-column multinomial with the same L2 penalty is a different estimator.
    tests/test_core.py checks agreement with sklearn.
    """
    import torch
    dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
    dt = torch.float32
    X = torch.as_tensor(Xtr, dtype=dt, device=dev)
    V = torch.as_tensor(Xva, dtype=dt, device=dev)
    y = torch.as_tensor(ytr, dtype=torch.long, device=dev)
    n, d = X.shape
    lam = 1.0 / (2.0 * C * n)
    binary = n_classes == 2
    out_dim = 1 if binary else n_classes
    W = torch.zeros(d, out_dim, dtype=dt, device=dev, requires_grad=True)
    b = torch.zeros(out_dim, dtype=dt, device=dev, requires_grad=True)
    opt = torch.optim.LBFGS([W, b], lr=1.0, max_iter=max_iter, history_size=20,
                            tolerance_grad=1e-6, tolerance_change=1e-9,
                            line_search_fn="strong_wolfe")
    yf = y.to(dt)

    def closure():
        opt.zero_grad()
        z = X @ W + b
        if binary:
            loss = torch.nn.functional.binary_cross_entropy_with_logits(z[:, 0], yf)
        else:
            loss = torch.nn.functional.cross_entropy(z, y)
        loss = loss + lam * (W * W).sum()
        loss.backward()
        return loss

    opt.step(closure)
    with torch.no_grad():
        z = (V @ W + b).double().cpu().numpy()
    if binary:
        z = np.concatenate([-z / 2.0, z / 2.0], axis=1)
    return z


def per_fold_logits(X: np.ndarray, y: np.ndarray, masks: list[np.ndarray],
                    C: float = 1.0, max_iter: int = 5000,
                    backend: str = "sklearn") -> np.ndarray:
    """Held-out logits for every input under the per-fold protocol.

    For each fold: fit StandardScaler on the training partition only, fit a
    logistic probe on the training partition only, predict logits for the
    held-out partition only. Concatenate into one (n, C) matrix aligned to the
    original input order.

    backend: "sklearn" (reference) or "torch" (same objective, L-BFGS on GPU).
    """
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y)
    n, _ = X.shape
    classes = np.unique(y)
    Z = np.full((n, len(classes)), np.nan)
    cls_to_col = {c: i for i, c in enumerate(classes)}
    y_idx = np.array([cls_to_col[c] for c in y])
    for m in masks:
        sc = StandardScaler().fit(X[~m])
        Xtr, Xva = sc.transform(X[~m]), sc.transform(X[m])
        if backend == "torch":
            present = np.unique(y_idx[~m])
            if len(present) != len(classes):
                raise RuntimeError("a class is missing from a training fold; "
                                   "use more data or fewer folds")
            Z[m] = _torch_probe(Xtr, y_idx[~m], Xva, len(classes), C=C,
                                max_iter=min(max_iter, 1000))
            continue
        probe = LogisticRegression(C=C, max_iter=max_iter, solver="lbfgs")
        probe.fit(Xtr, y[~m])
        lg = probe.decision_function(Xva)
        if lg.ndim == 1:                       # binary: decision_function gives one column
            lg = np.stack([-lg / 2.0, lg / 2.0], axis=1)
        cols = [cls_to_col[c] for c in probe.classes_]
        Z[np.ix_(np.where(m)[0], cols)] = lg
    if np.isnan(Z).any():
        raise RuntimeError("some inputs never received held-out logits")
    return Z


def per_fold_probs(X, y, masks, **kw) -> np.ndarray:
    """Held-out class probabilities (for probability-averaging baselines)."""
    Z = per_fold_logits(X, y, masks, **kw)
    return Z  # caller uses softmax(Z); kept separate for clarity in the pipeline


def softmax(Z: np.ndarray, axis: int = -1) -> np.ndarray:
    Z = Z - Z.max(axis=axis, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=axis, keepdims=True)


def preds_from_logits(Z: np.ndarray) -> np.ndarray:
    return np.asarray(Z).argmax(axis=1)


def accuracy(Z: np.ndarray, y: np.ndarray) -> float:
    return float((preds_from_logits(Z) == np.asarray(y)).mean())

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/fusion.py
"""Fusion rules used to measure marginal utility.

The class-conditional method (Eq. 2 of the rejected paper) is retained as the
primary rule, but the paper's own Appendix I showed the null is invariant across
fusion rules, so we compute all three by default and report the spread. Keeping
the parameter-free rules front and centre also pre-empts the objection that the
learned fusion can down-weight a candidate into invisibility.

Order sensitivity is real: class weights are initialised with mass on the first
teacher. Every call therefore sorts teacher keys canonically (alphabetically)
unless an explicit order is passed, so results are reproducible.
"""

from __future__ import annotations

import numpy as np

from folds import softmax


def fuse_mean_logits(Zs: list[np.ndarray]) -> np.ndarray:
    """Zero-parameter logit averaging. Standardise each teacher's logits first,
    otherwise a teacher with larger logit scale dominates by construction."""
    std = []
    for Z in Zs:
        Z = np.asarray(Z, dtype=np.float64)
        s = Z.std(axis=0, keepdims=True)
        s[s == 0] = 1.0
        std.append((Z - Z.mean(axis=0, keepdims=True)) / s)
    return np.mean(std, axis=0)


def fuse_mean_probs(Zs: list[np.ndarray]) -> np.ndarray:
    """Zero-parameter probability averaging (returns probabilities, not logits)."""
    return np.mean([softmax(np.asarray(Z, dtype=np.float64)) for Z in Zs], axis=0)


class ClassConditionalFusion:
    """Eq. 2: z_fused(c) = sum_k alpha_{c,k} * z_k(c) / tau_k + b_c.

    Parameter count = C*K + K + C (11-31 in the pathology experiments).
    Trained with cross-entropy + entropy regulariser (lambda=0.01) via Adam if
    torch is available, otherwise by scipy L-BFGS on the same objective. The
    scipy path is what makes this runnable on a Colab CPU cell and in CI.
    """

    def __init__(self, lam: float = 0.01, max_iter: int = 400, seed: int = 0):
        self.lam = lam
        self.max_iter = max_iter
        self.seed = seed

    def _unpack(self, theta, K, C):
        W = theta[: C * K].reshape(C, K)
        tau_hat = theta[C * K : C * K + K]
        b = theta[C * K + K :]
        return W, np.exp(np.clip(tau_hat, -10, 10)), b

    def _fused(self, theta, Zs, K, C):
        W, tau, b = self._unpack(theta, K, C)
        alpha = softmax(W, axis=1)                      # (C, K)
        out = np.zeros((Zs[0].shape[0], C))
        for k in range(K):
            out += alpha[:, k][None, :] * (Zs[k] / tau[k])
        return out + b[None, :]

    def _loss_grad(self, theta, Zs, y, K, C):
        """Objective: mean cross-entropy minus lam * mean per-class entropy.

        Gradient is analytic; tests/test_gradient.py checks it against a
        central-difference approximation before this is ever used on real data.
        """
        W, tau, b = self._unpack(theta, K, C)
        alpha = softmax(W, axis=1)                       # (C, K)
        Z = self._fused(theta, Zs, K, C)
        n = Z.shape[0]

        Zsh = Z - Z.max(axis=1, keepdims=True)
        logp = Zsh - np.log(np.exp(Zsh).sum(axis=1, keepdims=True))
        p = np.exp(logp)
        ce = -logp[np.arange(n), y].mean()
        ent = -(alpha * np.log(alpha + 1e-12)).sum(axis=1).mean()
        loss = ce - self.lam * ent

        onehot = np.zeros_like(p)
        onehot[np.arange(n), y] = 1.0
        dZ = (p - onehot) / n                            # dL/dZ_fused, (n, C)

        db = dZ.sum(axis=0)                              # (C,)
        dalpha = np.zeros((C, K))
        dtau_hat = np.zeros(K)
        for k in range(K):
            dalpha[:, k] = (dZ * (Zs[k] / tau[k])).sum(axis=0)
            # d/d tau_hat_k of alpha_k * Z_k * exp(-tau_hat_k) = -alpha_k * Z_k / tau_k
            dtau_hat[k] = -(dZ * (alpha[:, k][None, :] * Zs[k] / tau[k])).sum()

        def softmax_jac_rows(g_alpha):
            """Map dL/dalpha -> dL/dW row-wise through the per-class softmax."""
            out = np.zeros_like(g_alpha)
            for c in range(C):
                a = alpha[c]
                out[c] = a * (g_alpha[c] - np.dot(a, g_alpha[c]))
            return out

        dW = softmax_jac_rows(dalpha)
        # entropy term: ent = -(1/C) sum_c sum_k alpha_ck log alpha_ck
        d_ent_dalpha = -(np.log(alpha + 1e-12) + 1.0) / C
        dW = dW - self.lam * softmax_jac_rows(d_ent_dalpha)

        g = np.concatenate([dW.ravel(), dtau_hat, db])
        return loss, g

    def fit(self, Zs: list[np.ndarray], y: np.ndarray):
        Zs = [np.asarray(Z, dtype=np.float64) for Z in Zs]
        K = len(Zs)
        C = Zs[0].shape[1]
        rng = np.random.default_rng(self.seed)
        # W is a LOG-space parameter: alpha = softmax(W). Initialise near zero so
        # alpha starts close to uniform (1/K) and the softmax is not saturated.
        # Saturating it here zeroes the gradient and the fit silently does nothing.
        theta0 = np.concatenate([
            rng.normal(0.0, 0.01, size=C * K),
            np.zeros(K),          # tau_hat = 0  =>  tau = 1
            np.zeros(C),          # b = 0
        ])
        from scipy.optimize import minimize
        res = minimize(self._loss_grad, theta0, args=(Zs, y, K, C), jac=True,
                       method="L-BFGS-B", options={"maxiter": self.max_iter})
        self.theta_, self.K_, self.C_, self.res_ = res.x, K, C, res
        return self

    def predict_logits(self, Zs: list[np.ndarray]) -> np.ndarray:
        Zs = [np.asarray(Z, dtype=np.float64) for Z in Zs]
        return self._fused(self.theta_, Zs, self.K_, self.C_)

    @property
    def n_params(self) -> int:
        return self.C_ * self.K_ + self.K_ + self.C_

    @property
    def class_weights(self) -> np.ndarray:
        return softmax(self._unpack(self.theta_, self.K_, self.C_)[0], axis=1)


class IdentifiedClassConditionalFusion:
    """RECOMMENDED parameterisation: z_fused(c) = sum_k v_{c,k} z_k(c) + b_c.

    Why this replaces the v2 parameterisation as the default (corrected v3 note).

    v2 wrote the fused logit as alpha_{c,k} * z_k(c) / tau_k with alpha a
    per-class softmax. There is NO global scaling symmetry in that model
    (alpha is constrained to the simplex, so alpha -> c*alpha is not allowed,
    and rescaling tau alone changes the logits). What IS true:

      * The quantity that acts on the data is the effective weight
        v_{c,k} = alpha_{c,k} / tau_k. A teacher can carry a large alpha and
        still contribute almost nothing if its tau is large, so alpha alone
        must not be reported as "trust" (v2 Appendix G did this).
      * The decomposition (alpha, tau) is not identified when C < K
        (e.g. MHIST: 2 classes, 3 teachers), and it is weakly identified at the
        boundary where a useless teacher's tau -> infinity, because that
        teacher's alpha is then free. tests/test_core.py
        shows both cases.
      * The entropy regulariser acts on alpha, so where alpha is free it pins
        alpha towards uniform without changing the fit.

    Fitting v directly removes all three problems. Calibration is handled as a
    separate post-hoc step (v2 showed the learned temperatures do not beat
    temperature scaling). Parameter count is C*K + C (28 on BRACS with K=3,
    versus C*K + K + C = 31 for v2).
    """

    def __init__(self, ridge: float = 1e-3, max_iter: int = 500, seed: int = 0):
        self.ridge = ridge
        self.max_iter = max_iter
        self.seed = seed

    def _fused(self, theta, Zs, K, C):
        V = theta[: C * K].reshape(C, K)
        b = theta[C * K:]
        out = np.zeros((Zs[0].shape[0], C))
        for k in range(K):
            out += V[:, k][None, :] * Zs[k]
        return out + b[None, :]

    def _loss_grad(self, theta, Zs, y, K, C):
        V = theta[: C * K].reshape(C, K)
        b = theta[C * K:]
        Z = self._fused(theta, Zs, K, C)
        n = Z.shape[0]
        Zsh = Z - Z.max(axis=1, keepdims=True)
        logp = Zsh - np.log(np.exp(Zsh).sum(axis=1, keepdims=True))
        p = np.exp(logp)
        loss = -logp[np.arange(n), y].mean() + self.ridge * float((V ** 2).sum())

        onehot = np.zeros_like(p)
        onehot[np.arange(n), y] = 1.0
        dZ = (p - onehot) / n
        dV = np.zeros((C, K))
        for k in range(K):
            dV[:, k] = (dZ * Zs[k]).sum(axis=0)
        dV += 2 * self.ridge * V
        g = np.concatenate([dV.ravel(), dZ.sum(axis=0)])
        return loss, g

    def fit(self, Zs: list, y):
        Zs = [np.asarray(Z, dtype=np.float64) for Z in Zs]
        K, C = len(Zs), Zs[0].shape[1]
        rng = np.random.default_rng(self.seed)
        # initialise at the equal-weight average, which is a sensible starting point
        theta0 = np.concatenate([np.full(C * K, 1.0 / K) + rng.normal(0, 1e-3, C * K),
                                 np.zeros(C)])
        from scipy.optimize import minimize
        res = minimize(self._loss_grad, theta0, args=(Zs, y, K, C), jac=True,
                       method="L-BFGS-B", options={"maxiter": self.max_iter})
        self.theta_, self.K_, self.C_, self.res_ = res.x, K, C, res
        return self

    def predict_logits(self, Zs: list):
        Zs = [np.asarray(Z, dtype=np.float64) for Z in Zs]
        return self._fused(self.theta_, Zs, self.K_, self.C_)

    @property
    def n_params(self) -> int:
        return self.C_ * self.K_ + self.C_

    @property
    def class_weights(self) -> np.ndarray:
        """Identified per-class teacher weights: V normalised to sum to one per
        class. Safe to report as "trust placed in teacher k for class c"."""
        V = self.theta_[: self.C_ * self.K_].reshape(self.C_, self.K_)
        V = np.clip(V, 0.0, None)
        s = V.sum(axis=1, keepdims=True)
        s[s == 0] = 1.0
        return V / s

    @property
    def raw_class_weights(self) -> np.ndarray:
        return self.theta_[: self.C_ * self.K_].reshape(self.C_, self.K_)


RULES = {
    "mean_logits": lambda Zs, ytr=None, seed=0: fuse_mean_logits(Zs),
    "mean_probs": lambda Zs, ytr=None, seed=0: fuse_mean_probs(Zs),
    # default: the identified parameterisation (see IdentifiedClassConditionalFusion)
    "class_conditional": lambda Zs, ytr, seed=0: IdentifiedClassConditionalFusion(seed=seed).fit(Zs, ytr).predict_logits(Zs),
    "class_conditional_v2_legacy": lambda Zs, ytr, seed=0: ClassConditionalFusion(lam=0.01, seed=seed).fit(Zs, ytr).predict_logits(Zs),
}


def fuse(Zs, rule: str, ytr=None, seed: int = 0):
    if rule not in RULES:
        raise KeyError(f"unknown fusion rule {rule!r}; choose from {sorted(RULES)}")
    return RULES[rule](Zs, ytr=ytr, seed=seed)

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/pairs.py
"""Pair construction: the design decisions that determine whether the study has power.

The rejected version's fatal structural weakness was that it produced exactly
THREE matched comparisons, one of which (MHIST, CKA gap 0.012) had no separation
at all. Everything here is built to produce as many usable matched pairs as the
pool allows, and to record *why* each excluded pair was excluded, so the selection
is auditable rather than looking hand-picked.

Definitions (unchanged from v2 where it was sound):
  anchor   : reference encoder, default = highest individual held-out accuracy.
  base     : anchor + the median-CKA pool member, fixed BEFORE any pair is chosen
             (the base member is therefore not available as a candidate).
  marginal utility of candidate c, per fold f:
             U(c, f) = acc( fuse(base_f + c_f) ) - acc( fuse(base_f) )
  matched pair (a, b): |indiv_acc(a) - indiv_acc(b)| <= comp_tol
                       and |CKA(a, anchor) - CKA(b, anchor)| >= cka_min
  primary per-pair estimand:
             D(pair) = mean_f [ U(higher-CKA member, f) - U(lower-CKA member, f) ]
"""

from __future__ import annotations

from dataclasses import dataclass, field, asdict

import numpy as np


@dataclass
class PairSpec:
    pool: str
    anchor: str
    base: tuple
    high: str          # member with the higher CKA to the anchor
    low: str           # member with the lower CKA to the anchor
    cka_high: float
    cka_low: float
    cka_gap: float
    acc_high: float
    acc_low: float
    comp_gap: float
    category: str = "matched"

    def to_dict(self):
        d = asdict(self)
        d["base"] = list(self.base)
        return d


def choose_anchor(cka_with: dict[str, float], indiv_acc: dict[str, float],
                  rule: str = "best") -> str:
    """rule: 'best' = highest individual accuracy (default, matches v2),
             'median_cka' = the pool member with median CKA to the rest."""
    if rule == "best":
        return max(indiv_acc, key=lambda k: indiv_acc[k])
    if rule == "median_cka":
        names = sorted(cka_with)
        order = sorted(names, key=lambda k: cka_with[k])
        return order[len(order) // 2]
    raise ValueError(f"unknown anchor rule {rule!r}")


def fix_base(anchor: str, cka_with: dict[str, float]) -> tuple[str, str]:
    """Anchor plus the median-CKA pool member, chosen before any pair is formed.

    v3 FIX: the previous version picked others[len//2] in ALPHABETICAL order when
    the number of non-anchor members was odd, so the "median-CKA member" was in
    fact the alphabetical median. The base now always uses the CKA ordering
    (lower median for an even count; ties broken by name for determinism).
    """
    others = [k for k in cka_with if k != anchor]
    if not others:
        raise ValueError("pool must contain at least one non-anchor member")
    order = sorted(others, key=lambda k: (cka_with[k], k))
    mid = order[(len(order) - 1) // 2]
    return (anchor, mid)


def enumerate_pairs(pool: str, cka_with: dict[str, float], indiv_acc: dict[str, float],
                    comp_tol: float = 1.5, cka_min: float = 0.15,
                    anchor: str | None = None, cka_tol: float | None = None) -> list[PairSpec]:
    """List every candidate pair in the pool with its gaps and category.

    comp_tol : max competence gap (accuracy points) for a pair to be 'matched'.
               v2 used 5 points on general vision and reported 0.63-1.36 in
               pathology; 1.5 is the tighter default now that pools are large.
    cka_min  : min CKA gap for a pair to be informative at all. Pairs below it
               are 'no_separation' and excluded -- this is what MHIST was.
    cka_tol  : if given, the CKA gap must also exceed this many subsampling SDs
               of the CKA estimate. STRONGLY recommended: it is the check whose
               absence let a 0.012 gap be reported alongside a 0.259 gap.
    """
    if anchor is None:
        anchor = choose_anchor(cka_with, indiv_acc)
    base = fix_base(anchor, cka_with)
    cands = [k for k in cka_with if k not in base]

    specs: list[PairSpec] = []
    for i in range(len(cands)):
        for j in range(i + 1, len(cands)):
            a, b = cands[i], cands[j]
            hi, lo = (a, b) if cka_with[a] >= cka_with[b] else (b, a)
            cka_gap = abs(cka_with[a] - cka_with[b])
            comp_gap = abs(indiv_acc[a] - indiv_acc[b])

            if cka_tol is not None and cka_gap < cka_tol:
                cat = "no_separation"
            elif cka_gap < cka_min:
                cat = "cka_gap_too_small"
            elif comp_gap <= comp_tol:
                cat = "matched"
            else:
                cat = "unmatched"

            specs.append(PairSpec(
                pool=pool, anchor=anchor, base=base, high=hi, low=lo,
                cka_high=cka_with[hi], cka_low=cka_with[lo], cka_gap=cka_gap,
                acc_high=indiv_acc[hi], acc_low=indiv_acc[lo], comp_gap=comp_gap,
                category=cat,
            ))
    return specs


def matched_pairs(pool: str, cka_with, indiv_acc, **kw) -> list[PairSpec]:
    return [p for p in enumerate_pairs(pool, cka_with, indiv_acc, **kw)
            if p.category == "matched"]


def primary_pair(specs: list[PairSpec]) -> PairSpec | None:
    """The reported pair per pool: matched, maximising the CKA gap.

    Declaring this rule in advance is what makes Table 9-style auditing possible
    and defeats the 'chosen for its result' objection.
    """
    m = [p for p in specs if p.category == "matched"]
    return max(m, key=lambda p: p.cka_gap) if m else None


def summarise_pool(specs: list[PairSpec]) -> dict:
    """Auditable counts of why pairs were kept or dropped, per pool."""
    out: dict[str, int] = {}
    for p in specs:
        out[p.category] = out.get(p.category, 0) + 1
    out["total"] = len(specs)
    m = [p for p in specs if p.category == "matched"]
    out["matched_cka_gap_min"] = float(min((p.cka_gap for p in m), default=np.nan))
    out["matched_cka_gap_max"] = float(max((p.cka_gap for p in m), default=np.nan))
    out["matched_cka_gap_median"] = float(np.median([p.cka_gap for p in m])) if m else np.nan
    return out

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/inference.py
"""Inference utilities sized for the actual design.

The rejected version's primary interval, +0.25 [ -1.00, +1.50 ], was wide purely
because G = 3 clusters. Every function here takes a cluster label so the same
code gives you a tight interval at G = 16 without any change in analysis plan.
Implementations are deliberately hand-rolled (numpy/scipy only) so they run in
CI and on a Colab CPU cell.

CR1 small-sample correction and a t(G-1) reference are used throughout, matching
what the paper already adopted and what reviewers accepted as conservative.
"""

from __future__ import annotations

import numpy as np
from scipy import stats


def _design(x: np.ndarray | None, n: int) -> np.ndarray:
    if x is None:
        return np.ones((n, 1))
    x = np.asarray(x, dtype=np.float64).reshape(n, -1)
    return np.hstack([np.ones((n, 1)), x])


def cluster_robust_ols(y: np.ndarray, clusters: np.ndarray,
                       x: np.ndarray | None = None,
                       weights: np.ndarray | None = None) -> dict:
    """OLS with a cluster-robust (CR1) covariance and a t(G-1) reference.

    y       : outcome per observation (e.g. per-fold utility difference)
    clusters: cluster id per observation (e.g. dataset or pool)
    x       : optional regressor(s); None => intercept-only (the primary estimand)
    weights : optional per-observation weights (e.g. inverse-variance)

    Returns dict with estimate(s), SEs, t, df, p, and 95% CIs.
    """
    y = np.asarray(y, dtype=np.float64)
    clusters = np.asarray(clusters)
    n = y.shape[0]
    X = _design(x, n)
    w = np.ones(n) if weights is None else np.asarray(weights, dtype=np.float64)

    Xw = X * w[:, None]
    XtX_inv = np.linalg.pinv(Xw.T @ X)
    beta = XtX_inv @ (Xw.T @ y)
    resid = y - X @ beta

    G = len(np.unique(clusters))
    if G < 2:
        raise ValueError("need at least 2 clusters for a cluster-robust SE")
    meat = np.zeros((X.shape[1], X.shape[1]))
    for g in np.unique(clusters):
        m = clusters == g
        s = (Xw[m].T @ resid[m])
        meat += np.outer(s, s)
    scale = G / (G - 1) * (n - 1) / max(n - X.shape[1], 1)   # CR1
    V = scale * XtX_inv @ meat @ XtX_inv

    se = np.sqrt(np.diag(V))
    df = G - 1
    t = beta / se
    p = 2 * stats.t.sf(np.abs(t), df)
    tcrit = stats.t.ppf(0.975, df)
    return {
        "estimate": beta, "se": se, "t": t, "df": df, "p": p,
        "ci_low": beta - tcrit * se, "ci_high": beta + tcrit * se,
        "n": n, "G": G, "crit": tcrit,
    }


def tost_equivalence(y: np.ndarray, clusters: np.ndarray, margin: float,
                     x: np.ndarray | None = None, param_index: int = 1) -> dict:
    """Two-one-sided-tests equivalence against +/- margin, cluster-robust.

    With G = 3 this failed (p = 0.061) in the rejected version. At G >= 12 the
    same point estimate should clear it, which upgrades the claim from
    "we failed to reject a difference" to "we establish equivalence within
    +/- margin" -- the difference between a weak null and a real result.
    """
    r = cluster_robust_ols(y, clusters, x=x)
    b = r["estimate"][param_index] if r["estimate"].size > 1 else r["estimate"][0]
    se = r["se"][param_index] if r["se"].size > 1 else r["se"][0]
    df = r["df"]
    t_lo = (b - (-margin)) / se
    t_hi = (b - (+margin)) / se
    p_lo = stats.t.sf(t_lo, df)     # H0: b <= -margin
    p_hi = stats.t.cdf(t_hi, df)    # H0: b >= +margin
    tcrit = stats.t.ppf(0.95, df)   # one-sided 5%
    return {
        "estimate": b, "se": se, "df": df, "margin": margin,
        "p_lower": float(p_lo), "p_upper": float(p_hi),
        "p_tost": float(max(p_lo, p_hi)),
        "equivalent": bool(max(p_lo, p_hi) < 0.05),
        "ci90_low": b - tcrit * se, "ci90_high": b + tcrit * se,
        "G": r["G"], "n": r["n"],
    }


def mixed_effects_random_intercept(y: np.ndarray, clusters: np.ndarray,
                                   x: np.ndarray | None = None) -> dict:
    """Intercept-only random-intercept model, reported as a robustness check.

    Uses statsmodels MixedLM (REML) when available, with a t(G-1) reference to
    match the primary analysis. Falls back to a method-of-moments estimate
    whose SE is Var(grand mean) = (sigma2_between + sigma2_within / n_bar) / G.
    (v3 fix: the previous fallback divided the within variance by n*G, which
    understated the SE.)
    """
    y = np.asarray(y, dtype=np.float64)
    clusters = np.asarray(clusters)
    uniq = np.unique(clusters)
    G, n = len(uniq), y.shape[0]
    df = G - 1
    tcrit = stats.t.ppf(0.975, df) if df > 0 else np.nan
    try:
        import warnings
        import statsmodels.api as sm
        codes = np.searchsorted(uniq, clusters)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            r = sm.MixedLM(y, np.ones((n, 1)), groups=codes).fit(reml=True)
        est, se = float(r.params[0]), float(r.bse[0])
        s2u = float(np.asarray(r.cov_re).ravel()[0])
        s2w = float(r.scale)
        method = "statsmodels MixedLM (REML)"
    except Exception as e:                                   # fallback
        means = np.array([y[clusters == g].mean() for g in uniq])
        sizes = np.array([np.sum(clusters == g) for g in uniq], dtype=float)
        est = float(np.average(means, weights=sizes))
        ss_between = float(np.sum(sizes * (means - est) ** 2))
        ss_within = float(sum(((y[clusters == g] - y[clusters == g].mean()) ** 2).sum()
                              for g in uniq))
        s2w = ss_within / max(n - G, 1)
        ms_between = ss_between / max(G - 1, 1)
        n0 = (n - np.sum(sizes ** 2) / n) / max(G - 1, 1)
        s2u = max((ms_between - s2w) / max(n0, 1e-12), 0.0)
        se = float(np.sqrt((s2u + s2w / sizes.mean()) / G))
        method = f"method of moments (statsmodels unavailable: {type(e).__name__})"
    t = est / se if se > 0 else np.nan
    return {
        "estimate": est, "se": se, "t": t, "df": df,
        "p": float(2 * stats.t.sf(abs(t), df)) if se > 0 and df > 0 else np.nan,
        "ci_low": est - tcrit * se, "ci_high": est + tcrit * se,
        "sigma2_between": s2u, "sigma2_within": s2w,
        "icc": float(s2u / (s2u + s2w)) if (s2u + s2w) > 0 else np.nan,
        "G": G, "n": n, "method": method,
    }


def mannwhitney_u(a: np.ndarray, b: np.ndarray) -> dict:
    """Two-sided Mann-Whitney U, used for matched-band comparisons."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) == 0 or len(b) == 0:
        return {"U": np.nan, "p": np.nan, "n_a": len(a), "n_b": len(b)}
    U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    return {"U": float(U), "p": float(p), "n_a": len(a), "n_b": len(b)}


def spearman(a: np.ndarray, b: np.ndarray) -> dict:
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 3:
        return {"rho": np.nan, "p": np.nan, "n": len(a)}
    rho, p = stats.spearmanr(a, b)
    return {"rho": float(rho), "p": float(p), "n": len(a)}


def paired_fold_test(d: np.ndarray) -> dict:
    """Naive fold-independent paired tests. Reported but NOT primary -- folds
    within a dataset share training data, which is why the cluster-robust figure
    is the one to quote. Keep it in the table for comparability with v2."""
    d = np.asarray(d, float)
    n = len(d)
    m, sd = d.mean(), d.std(ddof=1) if n > 1 else np.nan
    t = m / (sd / np.sqrt(n)) if sd and sd > 0 else np.nan
    p_t = float(2 * stats.t.sf(abs(t), n - 1)) if np.isfinite(t) else np.nan
    try:
        _, p_w = stats.wilcoxon(d)
        p_w = float(p_w)
    except ValueError:
        p_w = np.nan
    return {"mean": float(m), "sd": float(sd), "n": n, "t": float(t),
            "p_t": p_t, "p_wilcoxon": p_w,
            "wins": int(np.sum(d > 0)), "losses": int(np.sum(d < 0)),
            "ties": int(np.sum(d == 0))}

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/analysis.py
"""The analysis battery: every estimand the resubmission needs, in one place.

Each function takes a `results` record produced by run_pipeline.py and returns
plain dicts so the whole thing can be dumped to JSON and asserted by
verify_claims.py -- the same discipline that made the v2 release checkable.

The five estimands, in the order they should appear in the paper:

  A. matched_null        primary: does CKA predict marginal utility at matched
                         competence? cluster-robust, t(G-1).
  B. dose_response       does the effect GROW with the CKA gap? This is the test
                         v2 could not run (slope SE = 1.20 on 6 points).
  C. equivalence         TOST at +/- margin. Upgrades "failed to reject" to
                         "established equivalent", which needs G >= ~12.
  D. selection_regret    the positive result: what does following the CKA
                         heuristic COST, against competence-only and against
                         other pre-fusion criteria? Not vulnerable to "your CI
                         was too wide", because it is a measured cost, not a null.
  E. mechanism_on_real   does held-out disagreement predict utility at matched
                         competence in REAL pools? v2 could only show this
                         synthetically; the AE called that "not independent".
"""

from __future__ import annotations

import numpy as np

from inference import (cluster_robust_ols, mannwhitney_u, mixed_effects_random_intercept,
                       paired_fold_test, spearman, tost_equivalence)


# --------------------------------------------------------------------------
# A. primary matched-pair null
# --------------------------------------------------------------------------

def matched_null(records: list[dict], margin_pairs: bool = False) -> dict:
    """records: one dict per matched pair with keys
         pool, pair_id, diff (per-pair mean of fold-level utility differences),
         fold_diffs (list/array of per-fold differences), primary (bool)

    Cluster = pool. Observations = one per pair (default) so that multiple pairs
    from the same pool do not masquerade as independent datasets.
    """
    rows = [r for r in records if r.get("primary", True)] if margin_pairs else records
    if not rows:
        return {"n_pairs": 0, "G": 0, "error": "no matched pairs supplied"}
    y = np.array([float(r["diff"]) for r in rows])
    cl = np.array([r["pool"] for r in rows])
    G = len(np.unique(cl))
    out = {"n_pairs": len(rows), "G": G}

    cr = cluster_robust_ols(y, cl)
    # intercept-only model: expose scalars, not 1-element arrays, so downstream
    # JSON and comparisons are unambiguous.
    out["cluster_robust"] = {
        "estimate_pp": float(cr["estimate"][0]), "se": float(cr["se"][0]),
        "t": float(cr["t"][0]), "df": int(cr["df"]), "p": float(cr["p"][0]),
        "ci_low": float(cr["ci_low"][0]), "ci_high": float(cr["ci_high"][0]),
        "n": int(cr["n"]), "G": int(cr["G"]),
        "reference": f"t(G-1) = t({int(cr['df'])})",
    }
    if G >= 2:
        me = mixed_effects_random_intercept(y, cl)
        out["mixed_effects"] = {k: (float(v) if isinstance(v, (np.floating, float)) else v)
                                for k, v in me.items()}
    # fold-level descriptive statistics (NOT primary; folds within a pool share data)
    allf = np.concatenate([np.asarray(r["fold_diffs"], float) for r in rows]) \
        if all("fold_diffs" in r for r in rows) else np.array([])
    if allf.size:
        out["fold_level_naive"] = paired_fold_test(allf)
        out["fold_level_naive"]["WARNING"] = ("folds within a pool share training data; "
                                              "quote the cluster-robust figure")
    return out


# --------------------------------------------------------------------------
# B. dose-response
# --------------------------------------------------------------------------

def dose_response(records: list[dict]) -> dict:
    """Regress the matched-pair utility difference on the CKA gap.

    Under the diversity heuristic a real effect must GROW with the gap. v2 got
    slope +0.09 with SE 1.20 (p = 0.94) over gaps 0.012-0.572 -- i.e. no power.
    Report the minimum detectable effect alongside the slope so a reviewer can
    see the power directly instead of inferring it from a p-value.
    """
    y = np.array([float(r["diff"]) for r in records])
    x = np.array([float(r["cka_gap"]) for r in records])
    cl = np.array([r["pool"] for r in records])
    res = cluster_robust_ols(y, cl, x=x)
    slope, se = float(res["estimate"][1]), float(res["se"][1])
    mde = 2.8 * se                      # ~80% power at alpha=.05, two-sided
    return {
        "slope_pp_per_unit_cka": slope, "se": se, "t": float(res["t"][1]),
        "df": res["df"], "p": float(res["p"][1]),
        "ci_low": float(res["ci_low"][1]), "ci_high": float(res["ci_high"][1]),
        "mde_80pct_power": mde,
        "cka_gap_range": [float(x.min()), float(x.max())],
        "n_pairs": len(y), "G": res["G"],
        "spearman": spearman(x, y),
        "INTERPRETATION": ("slope CI excludes any effect large enough to matter"
                           if abs(res["ci_high"][1]) < 1.0 and abs(res["ci_low"][1]) < 1.0
                           else "design still under-powered for a 1 pp/unit effect"),
    }


# --------------------------------------------------------------------------
# C. equivalence
# --------------------------------------------------------------------------

def equivalence(records: list[dict], margins=(1.0, 0.5, 0.25)) -> dict:
    """TOST at several margins. Report the largest margin you clear and say so
    plainly; do not claim equivalence at a margin the test does not support
    (v2 correctly refused to at n = 3)."""
    y = np.array([float(r["diff"]) for r in records])
    cl = np.array([r["pool"] for r in records])
    out = {}
    for m in margins:
        r = tost_equivalence(y, cl, margin=m)
        out[f"margin_{m}"] = r
    cleared = [m for m in margins if out[f"margin_{m}"]["equivalent"]]
    out["largest_margin_cleared"] = max(cleared) if cleared else None
    out["G"] = len(np.unique(cl))
    out["NOTE"] = ("equivalence established" if cleared else
                   "equivalence NOT established; report as failure to reject only")
    return out


# --------------------------------------------------------------------------
# D. selection regret -- the positive result
# --------------------------------------------------------------------------

def _select_by(cands: list[dict], key: str, reverse: bool) -> str:
    """Return the NAME of the selected candidate.

    Returns a name, not a dict: an earlier version returned the dict and the
    caller then failed a dict-membership test with 'unhashable type: dict'.
    `reverse=True` selects the largest value of `key` (competence, disagreement,
    headroom); `reverse=False` the smallest (CKA -- i.e. most divergent).
    """
    fn = max if reverse else min
    return fn(cands, key=lambda c: float(c[key]))["name"]


# Keys each regret criterion reads, in order of preference. This indirection
# exists because of a real bug: the rule asked for "disagreement_zs" while
# run_pool emitted "disagreement", every pool raised KeyError, every pool was
# silently appended to skipped_pools, and the criterion simply vanished from the
# output table. The headline comparison of the paper was missing and nothing
# failed loudly. Any key set that resolves to nothing now raises instead.
CRITERION_KEYS = {
    "cka_diversity": ("cka",),
    "disagreement_with_base": ("disagreement",),
    "competence": ("indiv_acc", "acc"),
    "oracle_headroom": ("headroom_contrib", "headroom"),
}


def _resolve(cands: list[dict], keys: tuple) -> str:
    """The first key present in the candidate records; raise if none is."""
    for k in keys:
        if all(k in c and c[k] is not None for c in cands):
            return k
    raise KeyError(f"none of {keys} present in candidate records "
                   f"(available: {sorted(cands[0])})")


def selection_regret(pools: dict[str, list[dict]]) -> dict:
    """pools: {pool_name: [candidate dicts]} where each candidate dict has
         name, cka (with anchor), indiv_acc, disagreement, utility (mean
         realised marginal utility over folds), headroom_contrib

    Criteria compared, all computable BEFORE any fusion is fit:
      cka_diversity          least similar to the anchor. The heuristic under
                             test, and the ONLY criterion that needs no labels.
      disagreement_with_base most different held-out predictions from the base.
                             NOT label-free: the predictions come from linear
                             probes trained on labelled folds, so it uses the
                             same information as `competence`. Do not describe
                             it as zero-shot in the paper.
      competence             individually strongest candidate (labels).
      oracle_headroom        largest unique-correct fraction (labels).
    Regret is measured against the candidate with the largest realised utility.

    Headline numbers: mean regret in accuracy points, and top-1 agreement rate
    with the oracle. A measured regret is immune to the "your null was
    under-powered" objection that sank v2.
    """
    reverse = {"cka_diversity": False, "disagreement_with_base": True,
               "competence": True, "oracle_headroom": True}
    rules = {r: (lambda cs, r=r: _select_by(cs, _resolve(cs, CRITERION_KEYS[r]), reverse[r]))
             for r in CRITERION_KEYS}
    out: dict = {r: {"regret_pp": [], "top1": [], "pools": 0} for r in rules}
    per_pool = {}
    skipped = []
    for pool, cands in pools.items():
        # A pool is unusable for regret if it has <2 candidates with finite utility,
        # or if names are not unique. Skip loudly rather than raising, so one bad
        # pool cannot take down the whole battery.
        cands = [c for c in cands if np.isfinite(c.get("utility", np.nan)) and "name" in c]
        if len(cands) < 2:
            skipped.append({"pool": pool, "reason": f"only {len(cands)} usable candidates"})
            continue
        if len({c["name"] for c in cands}) != len(cands):
            skipped.append({"pool": pool, "reason": "duplicate candidate names"})
            continue
        by_name = {c["name"]: c for c in cands}
        best = max(c["utility"] for c in cands)
        best_name = max(cands, key=lambda c: c["utility"])["name"]
        per_pool[pool] = {"oracle_choice": best_name, "oracle_utility": best}
        for rname, fn in rules.items():
            try:
                pick = fn(cands)
            except (ValueError, KeyError) as e:
                skipped.append({"pool": pool, "reason": f"{rname}: {e}"})
                continue
            if pick not in by_name:                       # defensive: should not happen
                skipped.append({"pool": pool, "reason": f"{rname} picked unknown {pick!r}"})
                continue
            u = by_name[pick]["utility"]
            out[rname]["regret_pp"].append(best - u)
            out[rname]["top1"].append(1.0 if pick == best_name else 0.0)
            out[rname]["pools"] += 1
            per_pool[pool][rname] = pick

    summary = {}
    for rname, d in out.items():
        if not d["regret_pp"]:
            continue
        reg = np.array(d["regret_pp"])
        summary[rname] = {
            "mean_regret_pp": float(reg.mean()),
            "median_regret_pp": float(np.median(reg)),
            "p90_regret_pp": float(np.percentile(reg, 90)),
            "max_regret_pp": float(reg.max()),
            "top1_agreement_with_oracle": float(np.mean(d["top1"])),
            "zero_regret_fraction": float(np.mean(reg == 0)),
            "n_pools": d["pools"],
        }
    ck = summary.get("cka_diversity", {})
    di = summary.get("disagreement_with_base", {})
    return {
        "per_rule": summary,
        "per_pool": per_pool,
        "skipped_pools": skipped,
        "n_pools_scored": len(per_pool),
        "HEADLINE": (f"following CKA diversity costs {ck.get('mean_regret_pp', float('nan')):.2f} pp "
                     f"on average vs. the oracle and picks the right teacher "
                     f"{100 * ck.get('top1_agreement_with_oracle', float('nan')):.0f}% of the time; "
                     f"prediction disagreement costs {di.get('mean_regret_pp', float('nan')):.2f} pp "
                     f"and is right {100 * di.get('top1_agreement_with_oracle', float('nan')):.0f}%")
        if ck and di else "insufficient pools",
    }


# --------------------------------------------------------------------------
# E. mechanism on real pools
# --------------------------------------------------------------------------

def mechanism_on_real(records: list[dict]) -> dict:
    """records: one per candidate (not per pair), with keys
         pool, name, cka, indiv_acc, disagreement (fraction of held-out inputs
         on which the candidate's probe prediction differs from the base fusion;
         probes are trained on labelled folds), utility

    Two regressions, both cluster-robust by pool:
      utility ~ disagreement + indiv_acc   -> does the mechanism hold in real data?
      utility ~ cka + indiv_acc           -> does CKA add anything once you control
                                             for competence?
    v2 reported disagreement going the WRONG way on real pools because of the
    competence entanglement; adding the competence term and using the controlled
    pools is what fixes it.
    """
    y = np.array([float(r["utility"]) for r in records])
    cl = np.array([r["pool"] for r in records])
    dis = np.array([float(r["disagreement"]) for r in records])
    cka = np.array([float(r["cka"]) for r in records])
    acc = np.array([float(r["indiv_acc"]) for r in records])

    def pack(res, idx, name):
        return {"coef_pp_per_unit": float(res["estimate"][idx]),
                "se": float(res["se"][idx]), "t": float(res["t"][idx]),
                "p": float(res["p"][idx]), "df": res["df"],
                "ci": [float(res["ci_low"][idx]), float(res["ci_high"][idx])],
                "term": name}

    r_dis = cluster_robust_ols(y, cl, x=np.stack([dis, acc], axis=1))
    r_cka = cluster_robust_ols(y, cl, x=np.stack([cka, acc], axis=1))
    r_both = cluster_robust_ols(y, cl, x=np.stack([cka, dis, acc], axis=1))
    return {
        "utility_on_disagreement_controlling_competence": pack(r_dis, 1, "disagreement"),
        "competence_coef_in_that_model": pack(r_dis, 2, "indiv_acc"),
        "utility_on_cka_controlling_competence": pack(r_cka, 1, "cka"),
        "joint_model": {"cka": pack(r_both, 1, "cka"),
                        "disagreement": pack(r_both, 2, "disagreement"),
                        "indiv_acc": pack(r_both, 3, "indiv_acc")},
        "marginal_spearmans": {"cka_vs_utility": spearman(cka, y),
                               "disagreement_vs_utility": spearman(dis, y),
                               "competence_vs_utility": spearman(acc, y),
                               "cka_vs_disagreement": spearman(cka, dis)},
        "n_candidates": len(y), "G": len(np.unique(cl)),
    }


def matched_band_comparison(records: list[dict], band=(0.55, 0.75),
                            low_acc=0.70, high_acc=0.90,
                            split: str = "relative", min_per_arm: int = 5) -> dict:
    """The v2 synthetic result (within a matched-CKA band, competent candidates
    give +8.44 pp vs +0.04 for weak ones) reproduced on REAL encoders. This is
    the single most persuasive figure the resubmission can have, and v2 could
    only produce it on synthetic data."""
    in_band = [r for r in records if band[0] <= float(r["cka"]) <= band[1]]
    accs = np.array([float(r["indiv_acc"]) for r in in_band])

    if split == "relative":
        # Absolute thresholds (v2 used 0.70 / 0.90) silently produce an empty arm
        # whenever a pool's accuracies all sit above or below them -- it returned
        # weak n=0 on the first end-to-end run. Splitting at the in-band median
        # makes the comparison well-defined for any pool, at the cost of no longer
        # being comparable to v2's exact numbers. Report which you used.
        if len(accs) < 2 * min_per_arm:
            cut = None
        else:
            cut = float(np.median(accs))
        lo_t = hi_t = cut
    else:
        lo_t, hi_t, cut = low_acc, high_acc, None

    if cut is None and split == "relative":
        weak = strong = []
    elif split == "relative":
        weak = [float(r["utility"]) for r in in_band if float(r["indiv_acc"]) < lo_t]
        strong = [float(r["utility"]) for r in in_band if float(r["indiv_acc"]) > hi_t]
    else:
        weak = [float(r["utility"]) for r in in_band if float(r["indiv_acc"]) < low_acc]
        strong = [float(r["utility"]) for r in in_band if float(r["indiv_acc"]) > high_acc]

    mw = mannwhitney_u(np.array(strong), np.array(weak))
    return {
        "band": band, "n_in_band": len(in_band), "split": split, "cut": cut,
        "acc_thresholds": [lo_t, hi_t],
        "mean_utility_weak_pp": float(np.mean(weak)) if weak else np.nan,
        "mean_utility_strong_pp": float(np.mean(strong)) if strong else np.nan,
        "n_weak": len(weak), "n_strong": len(strong),
        "mannwhitney": mw,
        "NOTE": ("thresholds must be declared before looking at utilities"
                 if len(weak) >= min_per_arm and len(strong) >= min_per_arm
                 else f"band too sparse ({len(weak)} weak vs {len(strong)} strong); "
                      "widen the band or add pools before quoting a p-value"),
    }

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/run_pipeline.py
"""End-to-end pipeline over one pool.

Input : a dict of frozen feature matrices {member_name: (n, d)}, labels y,
        optional groups (patient/slide/institution) for grouped folds.
Output: a single JSON-serialisable record containing everything the analysis
        battery in analysis.py needs, plus every intermediate quantity so that
        verify_claims.py can assert any published number against a fresh run.

Deliberately CPU-only and dependency-light (numpy/scipy/sklearn): feature
extraction is the only GPU step and it is a separate script. That split is what
makes this runnable inside a Colab session that disconnects, and testable in CI.
"""

from __future__ import annotations

import json
import time

import numpy as np

import analysis
import folds as F
import fusion
import pairs as P
from cka import cka_subsampling_sd, linear_cka, pairwise_cka_matrix


def run_pool(pool_name: str, feats: dict, y: np.ndarray, groups=None,
             k: int = 5, seed: int = 0, n_cap_cka: int = 10000,
             n_cap_analysis: int | None = None,
             comp_tol: float = 1.5, cka_min: float = 0.15, cka_sd_floor: float = 3.0,
             fusion_rules=("mean_logits", "mean_probs", "class_conditional"),
             probe_backend: str = "torch", verbose: bool = True,
             return_logits: bool = False) -> dict:
    """Run the whole pipeline for one pool.

    n_cap_cka      : rows used for the pairwise CKA matrix. Linear CKA costs
                     O(n*di*dj), so 276 pairs at n=30000 is minutes on CPU; the
                     measured subsampling SD is 0.003-0.006, far below any gap we
                     interpret, so capping is a free saving.
    n_cap_analysis : rows used for probes, fusion and accuracy. Capping here
                     changes absolute accuracies slightly but NOT the matched-pair
                     differences, which is the estimand. Set it to keep a 24-member
                     pool inside Colab RAM and a session.
    """
    t0 = time.time()
    names = sorted(feats)
    y = np.asarray(y)
    if groups is not None:
        groups = np.asarray(groups)

    # Apply the analysis cap FIRST and to everything, so features, labels, groups
    # and all downstream quantities stay row-aligned. A fixed seed makes it
    # reproducible; the cap is recorded in the output.
    if n_cap_analysis is not None and y.shape[0] > n_cap_analysis:
        n_orig = y.shape[0]
        rng_cap = np.random.default_rng(seed)
        keep = np.sort(rng_cap.choice(n_orig, size=n_cap_analysis, replace=False))
        feats = {m: np.asarray(v)[keep] for m, v in feats.items()}
        y = y[keep]
        if groups is not None:
            groups = groups[keep]
        print(f"[cap] analysis subsampled {n_orig} -> {len(keep)} rows (seed {seed})")
    n = y.shape[0]

    # ---- 1. folds (grouped when group labels exist) --------------------
    masks = F.make_folds(y, groups=groups, k=k, seed=seed)

    # ---- 2. held-out logits per member --------------------------------
    Z = {}
    for i_m, m in enumerate(names):
        t_m = time.time()
        Z[m] = F.per_fold_logits(feats[m], y, masks, backend=probe_backend)
        if verbose:
            print(f"  [{pool_name}] probe {i_m + 1}/{len(names)} {m:18s} "
                  f"acc {F.accuracy(Z[m], y) * 100:6.2f}%  ({time.time() - t_m:5.1f}s)",
                  flush=True)
    indiv_acc = {m: F.accuracy(Z[m], y) * 100.0 for m in names}

    # ---- 3. CKA -------------------------------------------------------
    C, cka_names = pairwise_cka_matrix(feats, n_cap=n_cap_cka, seed=seed)
    pos = {nm: i for i, nm in enumerate(cka_names)}
    cka_full = {}
    for a in names:
        for b in names:
            cka_full[(a, b)] = cka_full[(b, a)] = float(C[pos[a], pos[b]])
    mean_cka = {m: float(np.mean([cka_full[(m, o)] for o in names if o != m]))
                for m in names}
    anchor = P.choose_anchor(mean_cka, indiv_acc)
    cka_with_anchor = {m: (0.0 if m == anchor else cka_full[(m, anchor)]) for m in names}

    # CKA measurement noise: the check whose absence let a 0.012 gap be treated
    # as meaningful in v2. Computed on a subsample, so it is cheap.
    idx = np.sort(np.random.default_rng(seed).choice(n, size=min(4000, n), replace=False))
    cka_sd = {}
    for m in names:
        if m == anchor:
            continue
        cka_sd[m] = cka_subsampling_sd(feats[m][idx], feats[anchor][idx],
                                       n_draws=20, n_sub=min(2000, len(idx)), seed=seed)
    # BUG GUARD: a zero or non-finite SD would silently disable the separation
    # check (median([]) is nan, and nan comparisons are always False, so EVERY
    # pair would pass as separated). Fall back to v2's measured subsampling SD of
    # 0.003-0.006 and record that a fallback was used.
    finite_sds = [v for v in cka_sd.values() if np.isfinite(v) and v > 1e-9]
    if finite_sds:
        cka_sd_used = float(np.median(finite_sds))
        cka_sd_fallback = False
    else:
        cka_sd_used = 0.005
        cka_sd_fallback = True
    cka_tol = cka_sd_floor * cka_sd_used

    # ---- 4. base, pairs ----------------------------------------------
    base = P.fix_base(anchor, cka_with_anchor)
    base_names = list(base)
    specs = P.enumerate_pairs(pool_name, cka_with_anchor, indiv_acc,
                              comp_tol=comp_tol, cka_min=cka_min,
                              anchor=anchor, cka_tol=cka_tol)

    # ---- 5. utility of every member w.r.t. the fixed base -------------
    # Computed per fold and per rule.
    #
    # COST NOTE (this is the expensive step, and it is worth understanding why):
    # for each fusion rule we fit base + every candidate, per fold. The base fit
    # is IDENTICAL across candidates, so it is cached -- without that cache the
    # base was refit once per candidate, which for a 24-member pool means ~22
    # redundant fits per fold per rule. On Colab that is the difference between
    # minutes and hours per pool.
    util: dict[str, dict[str, np.ndarray]] = {}
    base_acc_per_rule_fold: dict[str, np.ndarray] = {}
    for rule in fusion_rules:
        base_fold_acc = np.zeros(k)
        base_models: list = [None] * k
        for f, m in enumerate(masks):
            Ztr = [Z[b][~m] for b in base_names]
            Zva = [Z[b][m] for b in base_names]
            if rule == "class_conditional":
                model = fusion.IdentifiedClassConditionalFusion(seed=seed).fit(Ztr, y[~m])
                base_models[f] = model                      # cached, reused below
                out = model.predict_logits(Zva)
            else:
                out = fusion.fuse(Zva, rule)
            base_fold_acc[f] = (np.asarray(out).argmax(1) == y[m]).mean() * 100.0
        base_acc_per_rule_fold[rule] = base_fold_acc

        cand_fold_acc = {}
        for c in names:
            if c in base_names:
                continue
            acc = np.zeros(k)
            for f, m in enumerate(masks):
                members = base_names + [c]
                Ztr = [Z[b][~m] for b in members]
                Zva = [Z[b][m] for b in members]
                if rule == "class_conditional":
                    model = fusion.IdentifiedClassConditionalFusion(seed=seed).fit(Ztr, y[~m])
                    out = model.predict_logits(Zva)
                else:
                    out = fusion.fuse(Zva, rule)
                acc[f] = (np.asarray(out).argmax(1) == y[m]).mean() * 100.0
            cand_fold_acc[c] = acc
        util[rule] = {c: cand_fold_acc[c] - base_fold_acc for c in cand_fold_acc}

    primary_rule = "mean_logits" if "mean_logits" in fusion_rules else fusion_rules[0]

    # ---- 6. prediction disagreement with the base -----------------------
    # Uses held-out probe predictions (probes are trained on labelled folds),
    # so this is NOT a label-free criterion. CKA is the only label-free one.
    base_pred = fusion.fuse([Z[b] for b in base_names], primary_rule).argmax(1)
    disagreement = {c: float((F.preds_from_logits(Z[c]) != base_pred).mean())
                    for c in names if c not in base_names}

    # ---- 7. oracle headroom contribution ------------------------------
    base_correct = np.zeros(n, dtype=bool)
    for b in base_names:
        base_correct |= (F.preds_from_logits(Z[b]) == y)
    headroom_contrib = {}
    for c in names:
        if c in base_names:
            continue
        c_correct = (F.preds_from_logits(Z[c]) == y)
        # fraction of inputs the candidate uniquely rescues over the base
        headroom_contrib[c] = float((c_correct & ~base_correct).mean() * 100.0)
    oracle_acc = float(np.mean(base_correct | np.logical_or.reduce(
        [(F.preds_from_logits(Z[c]) == y) for c in names if c not in base_names]
        or [base_correct])) * 100.0)

    # ---- 8. per-pair estimands ----------------------------------------
    pair_records = []
    for s in specs:
        rec = s.to_dict()
        rec["cka_sd_high"] = cka_sd.get(s.high)
        rec["cka_sd_low"] = cka_sd.get(s.low)
        for rule in fusion_rules:
            uh = util[rule].get(s.high)
            ul = util[rule].get(s.low)
            if uh is None or ul is None:
                continue
            rec[f"fold_diffs_{rule}"] = (uh - ul).tolist()
            rec[f"diff_{rule}"] = float(np.mean(uh - ul))
        rec["fold_diffs"] = rec.get(f"fold_diffs_{primary_rule}", [])
        rec["diff"] = rec.get(f"diff_{primary_rule}", float("nan"))
        rec["fusion_rule"] = primary_rule
        pair_records.append(rec)

    prim = P.primary_pair(specs)
    for rec in pair_records:
        rec["primary"] = bool(prim is not None and rec["high"] == prim.high
                              and rec["low"] == prim.low)

    matched = [r for r in pair_records if r["category"] == "matched"
               and np.isfinite(r["diff"])]

    # ---- 9. candidate-level records for the mechanism analysis ---------
    cand_records = []
    for c in names:
        if c in base_names:
            continue
        cand_records.append({
            "pool": pool_name, "name": c,
            "cka": cka_with_anchor[c],
            "indiv_acc": indiv_acc[c],
            "disagreement": disagreement[c],
            "headroom_contrib": headroom_contrib[c],
            "utility": float(np.mean(util[primary_rule][c])),
            "utility_per_fold": util[primary_rule][c].tolist(),
        })

    out = {
        "pool": pool_name,
        "n_members": len(names), "n_inputs": n, "k_folds": k, "seed": seed,
        "n_cap_cka": n_cap_cka, "n_cap_analysis": n_cap_analysis,
        "grouped_folds": groups is not None,
        "probe_backend": probe_backend,
        "anchor": anchor, "base": base_names,
        "indiv_acc": indiv_acc,
        "cka_with_anchor": cka_with_anchor,
        "cka_sd_with_anchor": cka_sd,
        "cka_tolerance_used": cka_tol,
        "cka_sd_fallback_used": cka_sd_fallback,
        "cka_sd_median": cka_sd_used,
        "oracle_acc": oracle_acc,
        "base_fold_acc": {r: v.tolist() for r, v in base_acc_per_rule_fold.items()},
        "utility_per_fold": {r: {c: v.tolist() for c, v in util[r].items()}
                             for r in util},
        "pair_specs_summary": P.summarise_pool(specs),
        "pairs": pair_records,
        "matched_pairs": matched,
        "candidates": cand_records,
        "primary_fusion_rule": primary_rule,
        "fusion_rules": list(fusion_rules),
        "wall_seconds": round(time.time() - t0, 1),
    }
    if return_logits:
        # not JSON-serialisable on purpose: the caller saves these with np.savez
        out["_logits"] = Z
        out["_y"] = y
        out["_masks"] = np.stack(masks)
    return out


def analyse_pools(pool_records: list[dict], margins=(1.0, 0.5, 0.25)) -> dict:
    """Run the full battery across pools. This is what produces the paper's numbers."""
    matched = []
    for rec in pool_records:
        matched.extend(rec["matched_pairs"])
    cands = []
    for rec in pool_records:
        cands.extend(rec["candidates"])

    primary = [m for m in matched if m.get("primary")]
    pools_for_regret = {}
    for rec in pool_records:
        pools_for_regret[rec["pool"]] = rec["candidates"]

    def safe(name, fn, *args, need=None, **kw):
        """One estimand failing (e.g. too few pools with matched pairs for a
        cluster-robust SE) must not take down the others. It is reported as
        not estimable, with the reason, instead."""
        if need is not None and not need:
            return {"not_estimable": "insufficient data for this estimand"}
        try:
            return fn(*args, **kw)
        except (ValueError, np.linalg.LinAlgError, ZeroDivisionError) as e:
            return {"not_estimable": f"{type(e).__name__}: {e}"}

    out = {
        "n_matched_pairs_all": len(matched),
        "n_matched_pairs_primary": len(primary),
        "G_pools": len({m["pool"] for m in matched}),
        "A_matched_null_all_pairs": safe("A_all", analysis.matched_null, matched, need=matched),
        "A_matched_null_primary_only": safe("A_primary", analysis.matched_null, primary, need=primary),
        "B_dose_response": safe("B", analysis.dose_response, matched, need=len(matched) >= 3),
        "C_equivalence": safe("C", analysis.equivalence, matched, margins=margins, need=matched),
        "D_selection_regret": safe("D", analysis.selection_regret, pools_for_regret),
        "E_mechanism_on_real": safe("E", analysis.mechanism_on_real, cands, need=len(cands) >= 5),
        "F_matched_band": safe("F", analysis.matched_band_comparison, cands, need=len(cands) >= 5),
    }
    return out


def dump(obj, path):
    def default(o):
        if isinstance(o, (np.floating, np.integer)):
            return o.item()
        if isinstance(o, np.ndarray):
            return o.tolist()
        raise TypeError(str(type(o)))
    with open(path, "w") as fh:
        json.dump(obj, fh, indent=2, default=default)

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/synth_pool.py
"""Synthetic pool generator with EXPLICIT control over CKA and competence.

Two purposes, and the second saves Colab hours:
  1. Scientific: populate the (CKA x competence) plane by construction, which no
     public checkpoint pool does -- the structural reason v2's evidence was thin.
  2. Engineering: a stand-in pool with known ground truth, so the whole analysis
     pipeline is validated end-to-end BEFORE any GPU time is spent.

DESIGN NOTE -- why this differs from v2 Appendix L.
v2 controlled CKA via shared-block *width* (ds) with a common realisation S and
private blocks P_k. Reimplementing that here produced a CKA range of only
0.259-0.352 across a 12-member pool: with fixed total width d and independently
drawn nuisance vectors, the nuisance cross-terms dominate and the ds contrast is
washed out. Since v2 reported reaching CKA 0.08-0.95, its generator must have
scaled the blocks differently than a literal reading of the description implies
(the full parameterisation is in its Appendix L, which is not in the released PDF).

Rather than guess, this version parameterises the quantity that actually controls
linear CKA directly. For centred features F_k = s_k * S + p_k * P_k with S shared
(identical realisation across members) and P_k independent:

    CKA(F_i, F_j) = corr_i * corr_j,   corr_k = s_k^2 / (s_k^2 + p_k^2)

which is exact in the large-n limit when S and P are independent and isotropic.
So specifying corr_k specifies member k's CKA to a fully-shared reference, and
pairs multiply. Competence is controlled independently by the class-signal energy
in the private block. The two knobs are orthogonal by construction, which is what
the design needs -- and `calibrate()` below verifies it numerically rather than
asserting it.

Kept from v2 (correctly): R_k is a random orthogonal matrix applied per member.
Linear CKA is orthogonally invariant, so rotation CANNOT vary CKA -- which is why
v2 rightly rejected the reviewer's suggestion to control CKA by rotation -- and it
is applied only so that no member's informative directions are axis-aligned.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from cka import linear_cka


@dataclass
class SyntheticPool:
    n_classes: int
    n_per_class: int
    d_total: int
    members: list
    X: dict          # name -> (n, d) features
    y: np.ndarray
    corr: dict       # name -> shared-energy fraction actually realised
    sig: dict        # name -> private class-signal strength actually realised
    phi: float       # fraction of class-signal energy in the shared block
    truth: dict


def _class_signal(y: np.ndarray, n_classes: int, dim: int,
                  rng: np.random.Generator) -> np.ndarray:
    """Per-input class prototype signal in `dim` dimensions, unit total energy."""
    P = rng.normal(size=(n_classes, dim))
    P /= np.linalg.norm(P, axis=1, keepdims=True)
    return P[y] / np.sqrt(max(dim, 1))


def make_synthetic_pool(n_members: int = 24, n_classes: int = 7, n_per_class: int = 400,
                        d_total: int = 384, phi: float = 0.5, seed: int = 0,
                        corr_range=(0.15, 0.95), sig_range=(0.3, 2.5),
                        shared_dim_frac: float = 0.5,
                        cka_dependent_utility: bool = False) -> SyntheticPool:
    """Build a pool spanning CKA and competence near-independently.

    corr_range : range of the shared-energy fraction s^2/(s^2+p^2) per member.
                 Determines each member's CKA to the shared reference; pairwise
                 CKA is the product of the two members' corr values.
    sig_range  : private class-signal strength per member. Determines competence.
    phi        : fraction of total class-signal energy placed in the shared block.
                 This is the mediator v2 found: it sets how correlated members'
                 errors are, and it flips the sign of the CKA-disagreement
                 relation. It is unobservable to a practitioner with frozen features.
    cka_dependent_utility : PIPELINE VALIDATION SWITCH (see tests).
    """
    rng = np.random.default_rng(seed)
    n = n_classes * n_per_class
    y = np.repeat(np.arange(n_classes), n_per_class)
    rng.shuffle(y)

    d_shared = max(8, int(round(d_total * shared_dim_frac)))
    d_priv = max(8, d_total - d_shared)

    # one shared realisation, identical across members -> contributes maximally to CKA
    S_nuis = rng.normal(size=(n, d_shared))
    S_sig = _class_signal(y, n_classes, d_shared, rng)

    corr_target = rng.uniform(corr_range[0], corr_range[1], size=n_members)
    sig_target = rng.uniform(sig_range[0], sig_range[1], size=n_members)

    names = [f"synth_{i:02d}" for i in range(n_members)]
    X, corr, sig = {}, {}, {}
    for i, name in enumerate(names):
        c = float(corr_target[i])
        a = float(sig_target[i])
        # split total energy between shared and private blocks per corr
        s_w = np.sqrt(c)
        p_w = np.sqrt(max(1.0 - c, 1e-6))

        P_nuis = rng.normal(size=(n, d_priv))
        P_sig = _class_signal(y, n_classes, d_priv, rng)

        shared = s_w * (S_nuis + phi * np.sqrt(d_shared) * S_sig)
        private = p_w * (P_nuis + (1.0 - phi) * a * np.sqrt(d_priv) * P_sig)
        F = np.hstack([shared, private])

        # random orthogonal mixing: CKA-invariant, breaks axis alignment
        Q, _ = np.linalg.qr(rng.normal(size=(F.shape[1], F.shape[1])))
        X[name] = F @ Q
        corr[name], sig[name] = c, a

    truth = {
        "corr": corr, "sig": sig, "phi": phi,
        "d_shared": d_shared, "d_priv": d_priv,
        "cka_dependent_utility": bool(cka_dependent_utility),
        "expected_pairwise_cka": "corr_i * corr_j (large-n limit)",
        "NOTE": ("utility was made CKA-dependent by construction: the pipeline MUST "
                 "recover a positive dose-response slope" if cka_dependent_utility
                 else "utility carries no CKA signal: the pipeline MUST return a null"),
    }
    return SyntheticPool(n_classes=n_classes, n_per_class=n_per_class, d_total=d_total,
                         members=names, X=X, y=y, corr=corr, sig=sig, phi=phi, truth=truth)


def calibrate(n_members: int = 16, seed: int = 0, **kw) -> dict:
    """Verify the construction actually delivers the claimed control BEFORE using
    it for science. Returns the realised CKA range, the realised CKA vs. the
    predicted corr_i*corr_j, and the CKA-competence correlation.

    If realised CKA does not track corr_i*corr_j, or if CKA and competence come
    out collinear, the pool is confounded in the same way the real pools were and
    must not be used as a 'controlled' arm.
    """
    pool = make_synthetic_pool(n_members=n_members, seed=seed, **kw)
    names = pool.members
    realised, predicted = [], []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            realised.append(linear_cka(pool.X[names[i]], pool.X[names[j]]))
            predicted.append(pool.corr[names[i]] * pool.corr[names[j]])
    realised, predicted = np.array(realised), np.array(predicted)
    from inference import spearman
    return {
        "n_pairs": len(realised),
        "cka_range": [float(realised.min()), float(realised.max())],
        "corr_realised_vs_predicted": float(np.corrcoef(realised, predicted)[0, 1]),
        "rmse": float(np.sqrt(np.mean((realised - predicted) ** 2))),
        "spearman": spearman(realised, predicted),
        "VERDICT": ("construction delivers the claimed CKA control"
                    if realised.max() - realised.min() > 0.4
                    and np.corrcoef(realised, predicted)[0, 1] > 0.95
                    else "CONSTRUCTION FAILED: CKA not controlled -- do not use"),
    }


def inject_utility_signal(utilities: np.ndarray, cka_gaps: np.ndarray,
                          beta: float = 6.0, seed: int = 0) -> np.ndarray:
    """TEST-ONLY: add a known CKA-dependent component so the analysis has
    something to find. Used by the positive control in test_pipeline_synthetic.py."""
    rng = np.random.default_rng(seed)
    return np.asarray(utilities, float) + beta * np.asarray(cka_gaps, float) \
        + rng.normal(0, 0.05, size=len(utilities))

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/imgcache.py
"""Dataset manifests and decoded 224x224 uint8 image caches.

Why decode once: v2 spent ~19 min per encoder just decoding BRACS RoIs
(139-173 MP each) over Drive's FUSE mount. Decoding each image once to a
224x224 uint8 array and letting every encoder read that array makes the
per-encoder cost GPU-bound. Preprocessing is identical to v2's CTransPath
extraction (Resize(224) shorter side, CenterCrop(224)); each encoder then
applies only its own mean/std normalisation.

Row order is defined ONCE, by the manifest, and every encoder's features are
written in that order. The manifest stores a hash of (uid, label) so the
analysis can assert alignment instead of assuming it.
"""

from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import shutil
import subprocess  # noqa: F401  (kept for callers)
import time

import numpy as np

import config as CFG

IMG_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# Datasets fetched from the internet. Their decoded 224px caches are saved to
# Drive (CRC-VAL 1.1 GB, NCT 20k 3.0 GB, PCam 20k 3.0 GB) so a Colab reset never
# forces a re-download again. Set False if Drive space is tight.
DOWNLOADED = {"crc_val_7k", "nct_crc_100k", "pcam"}
PERSIST_DOWNLOADED = True


# ---------------------------------------------------------------------------
# small utilities
# ---------------------------------------------------------------------------

def _mk(*dirs):
    for d in dirs:
        os.makedirs(d, exist_ok=True)


def manifest_hash(uids, labels) -> str:
    h = hashlib.sha256()
    for u, l in zip(uids, labels):
        h.update(f"{u}\t{int(l)}\n".encode())
    return h.hexdigest()


def stratified_cap(labels, cap, seed=CFG.SEED, groups=None):
    """Indices (sorted) of a stratified subsample of size `cap`.
    If groups are given, whole groups are kept or dropped together so that the
    grouped folds downstream stay meaningful."""
    labels = np.asarray(labels)
    n = len(labels)
    if cap is None or n <= cap:
        return np.arange(n)
    rng = np.random.default_rng(seed)
    if groups is not None:
        groups = np.asarray(groups)
        ug = np.unique(groups)
        rng.shuffle(ug)
        keep, total = [], 0
        for g in ug:
            idx = np.where(groups == g)[0]
            if total + len(idx) > cap and total > 0:
                continue
            keep.append(idx)
            total += len(idx)
            if total >= cap * 0.98:
                break
        return np.sort(np.concatenate(keep))
    from sklearn.model_selection import train_test_split
    idx, _ = train_test_split(np.arange(n), train_size=cap, stratify=labels,
                              random_state=seed)
    return np.sort(idx)


def download(url, dst, retries=8, chunk=1 << 22):
    """Resumable download that reports progress IN THE NOTEBOOK.

    (The previous version used curl in a subprocess; Colab does not show a
    subprocess's progress bar, so an 11.7 GB download looked frozen.)
    A partial file is kept as <dst>.part and resumed with an HTTP Range request,
    so interrupting the cell loses nothing."""
    import requests
    if os.path.exists(dst) and os.path.getsize(dst) > 0 and not os.path.exists(dst + ".part"):
        return dst
    _mk(os.path.dirname(dst))
    part = dst + ".part"
    for attempt in range(retries):
        have = os.path.getsize(part) if os.path.exists(part) else 0
        headers = {"Range": f"bytes={have}-"} if have else {}
        try:
            with requests.get(url, stream=True, timeout=60, headers=headers,
                              allow_redirects=True) as r:
                if r.status_code == 416:                  # already complete
                    os.replace(part, dst)
                    return dst
                r.raise_for_status()
                if have and r.status_code != 206:          # server ignored Range: restart
                    have = 0
                clen = int(r.headers.get("Content-Length", 0) or 0)
                total = clen + have if clen else 0
                mode = "ab" if have else "wb"
                t0, last, got = time.time(), 0.0, have
                name = os.path.basename(dst)
                print(f"  downloading {name}: "
                      + (f"{total / 1e9:.2f} GB" if total > have else "size not reported")
                      + (f" (resuming at {have / 1e9:.2f} GB)" if have else ""), flush=True)
                with open(part, mode) as fh:
                    for blk in r.iter_content(chunk_size=chunk):
                        fh.write(blk)
                        got += len(blk)
                        now = time.time()
                        if now - last > 30:
                            rate = (got - have) / max(now - t0, 1e-9)
                            eta = (total - got) / max(rate, 1) / 60 if total else 0.0
                            print(f"    {got / 1e9:6.2f} / "
                                  + (f"{total / 1e9:.2f} GB  " if total > have else "? GB  ")
                                  + f"{rate / 1e6:6.1f} MB/s"
                                  + (f"  ETA {eta:5.1f} min" if total > have else ""), flush=True)
                            last = now
            if total and os.path.getsize(part) < total:
                raise IOError(f"incomplete: {os.path.getsize(part)} of {total} bytes")
            os.replace(part, dst)
            print(f"  downloaded {name} in {(time.time() - t0) / 60:.1f} min", flush=True)
            return dst
        except Exception as e:
            print(f"  download interrupted ({type(e).__name__}: {str(e)[:120]}); "
                  f"retry {attempt + 1}/{retries} in 15 s, resuming", flush=True)
            time.sleep(15)
    raise RuntimeError(f"download failed: {url}")


def _zip_tree(zp, inner):
    """Class-folder listing straight from the zip index (no extraction).
    Reproduces _folder_tree's ordering exactly: classes sorted, files sorted
    by name within a class. The manifest hash check guards this equivalence."""
    import zipfile
    by_cls = {}
    with zipfile.ZipFile(zp) as z:
        for n in z.namelist():
            parts = n.split("/")
            if len(parts) != 3 or parts[0] != inner or not parts[2]:
                continue
            if os.path.splitext(parts[2])[1].lower() not in IMG_EXT:
                continue
            by_cls.setdefault(parts[1], []).append(parts[2])
    classes = sorted(by_cls)
    root = os.path.join(CFG.LOCAL_DL, inner)
    files, labels = [], []
    for ci, c in enumerate(classes):
        for f in sorted(by_cls[c]):
            files.append(os.path.join(root, c, f))
            labels.append(ci)
    return classes, files, labels, root


def extract_needed(zp, files):
    """Extract only the members the manifest uses (20k of 100k for NCT)."""
    import zipfile
    missing = [f for f in files if not os.path.exists(f)]
    if not missing:
        return
    base = CFG.LOCAL_DL
    t0 = time.time()
    print(f"  extracting {len(missing)} of the zip's images that the manifest uses ...", flush=True)
    with zipfile.ZipFile(zp) as z:
        for i, f in enumerate(missing):
            z.extract(os.path.relpath(f, base).replace(os.sep, "/"), base)
            if (i + 1) % 5000 == 0:
                print(f"    {i + 1}/{len(missing)}", flush=True)
    print(f"  extracted in {(time.time() - t0) / 60:.1f} min", flush=True)


# ---------------------------------------------------------------------------
# manifests: each builder returns dict(uids, labels, groups|None, classes, reader)
#   reader = ("files", [paths]) | ("torchvision", spec) | ("h5", path, row_index)
# ---------------------------------------------------------------------------

def _bracs():
    root = CFG.BRACS_DIR
    classes = sorted(d for d in os.listdir(os.path.join(root, "train"))
                     if os.path.isdir(os.path.join(root, "train", d)))
    files, labels = [], []
    for split in ("train", "val"):                       # v2 order: train + val
        for ci, c in enumerate(classes):
            d = os.path.join(root, split, c)
            fs = sorted(f for f in os.listdir(d) if os.path.splitext(f)[1].lower() in IMG_EXT)
            files += [os.path.join(d, f) for f in fs]
            labels += [ci] * len(fs)
    pat = re.compile(r"^BRACS_(\d+)_")
    g = [pat.match(os.path.basename(f)) for f in files]
    if sum(m is None for m in g) > 0.01 * len(files):
        print("  [bracs] WARNING: WSI id not parseable from filenames -> ungrouped folds")
        groups = None
    else:
        groups = [("BRACS_" + m.group(1)) if m else os.path.basename(f) for m, f in zip(g, files)]
    uids = [os.path.relpath(f, root) for f in files]
    return dict(uids=uids, labels=labels, groups=groups, classes=classes, reader=("files", files))


def _sicapv2():
    import pandas as pd
    part = os.path.join(CFG.SICAP_DIR, "partition", "Test")
    cols = ["NC", "G3", "G4", "G5"]
    df = pd.concat([pd.read_excel(os.path.join(part, "Train.xlsx")),
                    pd.read_excel(os.path.join(part, "Test.xlsx"))], ignore_index=True)
    names = list(df["image_name"])
    labels = df[cols].values.argmax(1).tolist()
    groups = [n.split("_Block")[0] for n in names]
    files = [os.path.join(CFG.SICAP_DIR, "images", n) for n in names]
    return dict(uids=names, labels=labels, groups=groups, classes=cols, reader=("files", files))


def _mhist():
    import pandas as pd
    csvp = next((os.path.join(CFG.MHIST_DIR, c) for c in CFG.MHIST_CSV_CANDIDATES
                 if os.path.exists(os.path.join(CFG.MHIST_DIR, c))), None)
    if csvp is None:
        raise FileNotFoundError(f"no MHIST annotations csv in {CFG.MHIST_DIR}")
    df = pd.read_csv(csvp)
    classes = sorted(df["Majority Vote Label"].unique())          # ['HP', 'SSA']
    cmap = {c: i for i, c in enumerate(classes)}
    names = list(df["Image Name"])
    labels = df["Majority Vote Label"].map(cmap).tolist()
    files = [os.path.join(CFG.MHIST_DIR, "images", n) for n in names]
    return dict(uids=names, labels=labels, groups=None, classes=classes,
                reader=("files", files), extra={"partition": list(df["Partition"])})


def _folder_tree(root):
    classes = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))
    files, labels = [], []
    for ci, c in enumerate(classes):
        fs = sorted(f for f in os.listdir(os.path.join(root, c))
                    if os.path.splitext(f)[1].lower() in IMG_EXT)
        files += [os.path.join(root, c, f) for f in fs]
        labels += [ci] * len(fs)
    return classes, files, labels


def _zenodo_zip(fname, inner):
    zp = download(CFG.ZENODO_NCT.format(fname), os.path.join(CFG.LOCAL_DL, fname))
    classes, files, labels, root = _zip_tree(zp, inner)
    return dict(uids=[os.path.relpath(f, root) for f in files], labels=labels, groups=None,
                classes=classes, reader=("files", files), zip=zp)


def _crc_val_7k():
    return _zenodo_zip("CRC-VAL-HE-7K.zip", "CRC-VAL-HE-7K")


def _nct_crc_100k():
    return _zenodo_zip("NCT-CRC-HE-100K.zip", "NCT-CRC-HE-100K")


def _pcam():
    import gzip
    import pandas as pd
    out = {}
    for part in ("x", "y"):
        gz = download(CFG.ZENODO_PCAM.format(f"camelyonpatch_level_2_split_test_{part}.h5.gz"),
                      os.path.join(CFG.LOCAL_DL, f"pcam_test_{part}.h5.gz"))
        h5 = gz[:-3]
        if not os.path.exists(h5):
            print(f"  gunzip {os.path.basename(gz)} ...")
            with gzip.open(gz, "rb") as fi, open(h5 + ".tmp", "wb") as fo:
                shutil.copyfileobj(fi, fo, length=1 << 24)
            os.replace(h5 + ".tmp", h5)
        out[part] = h5
    meta = download(CFG.ZENODO_PCAM.format("camelyonpatch_level_2_split_test_meta.csv"),
                    os.path.join(CFG.LOCAL_DL, "pcam_test_meta.csv"))
    import h5py
    with h5py.File(out["y"], "r") as f:
        labels = np.asarray(f["y"]).reshape(-1).astype(int).tolist()
    wsi = list(pd.read_csv(meta)["wsi"])
    assert len(wsi) == len(labels), "PCam meta / label length mismatch"
    n = len(labels)
    return dict(uids=[f"pcam_test_{i}" for i in range(n)], labels=labels, groups=wsi,
                classes=["normal", "tumor"], reader=("h5", out["x"], list(range(n))))


class _HFImages:
    """A Hugging Face image dataset exposed like a torchvision dataset
    (__getitem__ -> (PIL image, label)), used as a fallback mirror."""

    def __init__(self, hf_id, split):
        from datasets import load_dataset
        self.ds = load_dataset(hf_id, split=split)
        self._labels = [int(v) for v in self.ds["label"]]
        self.classes = list(self.ds.features["label"].names)
        self.source = f"huggingface:{hf_id}:{split}"

    def __len__(self):
        return len(self._labels)

    def __getitem__(self, i):
        r = self.ds[int(i)]
        return r["image"], int(r["label"])


# Mirrors used only when the official torchvision host fails. A manifest built
# from a mirror records it (reader name "<dataset>_hf"), so every later session
# decodes from the same source and row order.
HF_MIRRORS = {"gtsrb": ("tanganke/gtsrb", "train")}   # 26,640 images, 43 classes (= torchvision train)


def _tv(name):
    """torchvision datasets: returns (dataset_without_transform, labels)."""
    if name.endswith("_hf"):
        hf_id, split = HF_MIRRORS[name[:-3]]
        ds = _HFImages(hf_id, split)
        return ds, ds._labels
    from torchvision import datasets as D
    root = os.path.join(CFG.LOCAL_DL, "tv")
    _mk(root)
    if name == "dtd":
        ds = D.DTD(root, split="train", download=True)
    elif name == "aircraft":
        ds = D.FGVCAircraft(root, split="train", download=True)
    elif name == "eurosat":
        ds = D.EuroSAT(root, download=True)
    elif name == "pets":
        ds = D.OxfordIIITPet(root, split="trainval", download=True)
    elif name == "food101":
        ds = D.Food101(root, split="train", download=True)
    elif name == "cifar100":
        ds = D.CIFAR100(root, train=True, download=True)
    elif name == "gtsrb":
        ds = D.GTSRB(root, split="train", download=True)
    elif name == "country211":
        ds = D.Country211(root, split="train", download=True)
    else:
        raise KeyError(name)
    labels = tv_labels(ds)
    return ds, labels


def tv_labels(ds):
    for attr in ("targets", "_labels"):
        if hasattr(ds, attr):
            return [int(v) for v in getattr(ds, attr)]
    if hasattr(ds, "_samples"):
        return [int(s[1]) for s in ds._samples]
    print("  (labels not exposed; iterating once)")
    return [int(ds[i][1]) for i in range(len(ds))]


def _tv_manifest(name):
    src = name
    try:
        ds, labels = _tv(name)
    except Exception as e:
        if name not in HF_MIRRORS:
            raise
        print(f"  [{name}] official torchvision download failed ({type(e).__name__}: "
              f"{str(e)[:100]}); using mirror {HF_MIRRORS[name][0]}", flush=True)
        src = name + "_hf"
        ds, labels = _tv(src)
    classes = [str(c) for c in getattr(ds, "classes", sorted(set(labels)))]
    return dict(uids=[f"{src}_{i}" for i in range(len(labels))], labels=labels, groups=None,
                classes=classes, reader=("torchvision", src, list(range(len(labels)))),
                extra_source=getattr(ds, "source", f"torchvision:{name}"))


BUILDERS = {
    "bracs": _bracs, "sicapv2": _sicapv2, "mhist": _mhist,
    "crc_val_7k": _crc_val_7k, "nct_crc_100k": _nct_crc_100k, "pcam": _pcam,
    **{k: (lambda k=k: _tv_manifest(k)) for k in
       ("dtd", "aircraft", "eurosat", "pets", "food101", "cifar100", "gtsrb", "country211")},
}


def dataset_cfg(name):
    for dsets in (CFG.PATHOLOGY_DATASETS, CFG.VISION_DATASETS):
        if name in dsets:
            return dsets[name]
    raise KeyError(name)


def build_manifest(name, force=False):
    """Build (or load) the manifest; applies the stratified cap. Saved to Drive."""
    _mk(CFG.MANIFEST_DIR)
    jp = os.path.join(CFG.MANIFEST_DIR, f"{name}.json")
    if os.path.exists(jp) and not force:
        with open(jp) as fh:
            man = json.load(fh)
        return man
    t0 = time.time()
    m = BUILDERS[name]()
    labels = np.asarray(m["labels"], int)
    groups = None if m["groups"] is None else np.asarray(m["groups"])
    cap = dataset_cfg(name)["cap"]
    keep = stratified_cap(labels, cap, groups=groups)
    reader = list(m["reader"])
    # subset the reader's per-row source list (last element)
    reader[-1] = [reader[-1][i] for i in keep]
    if m.get("zip"):
        extract_needed(m["zip"], reader[-1])
    man = dict(
        name=name,
        n=int(len(keep)), n_before_cap=int(len(labels)), cap=cap, seed=CFG.SEED,
        classes=list(m["classes"]),
        uids=[m["uids"][i] for i in keep],
        labels=labels[keep].tolist(),
        groups=None if groups is None else groups[keep].tolist(),
        reader=reader,
        class_counts=np.bincount(labels[keep], minlength=len(m["classes"])).tolist(),
        n_groups=None if groups is None else int(len(np.unique(groups[keep]))),
        built=time.strftime("%Y-%m-%d %H:%M:%S"),
    )
    if "extra" in m:
        man["extra"] = {k: [v[i] for i in keep] for k, v in m["extra"].items()}
    if "extra_source" in m:
        man["source"] = m["extra_source"]
    man["hash"] = manifest_hash(man["uids"], man["labels"])
    with open(jp, "w") as fh:
        json.dump(man, fh)
    with open(os.path.join(CFG.MANIFEST_DIR, f"{name}.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["row", "uid", "label", "group"])
        for i, (u, l) in enumerate(zip(man["uids"], man["labels"])):
            w.writerow([i, u, l, "" if man["groups"] is None else man["groups"][i]])
    print(f"  [{name}] manifest n={man['n']} (of {man['n_before_cap']}) "
          f"classes={len(man['classes'])} groups={man['n_groups']} "
          f"({time.time() - t0:.0f}s)")
    return man


# ---------------------------------------------------------------------------
# decoding to the uint8 cache
# ---------------------------------------------------------------------------

class _Decoder:
    """torch Dataset yielding uint8 (224, 224, 3) arrays for a manifest."""

    def __init__(self, man, size=CFG.IMAGE_SIZE):
        self.reader = man["reader"]
        self.size = size
        self._h5 = None
        self._tv = None

    def __len__(self):
        return len(self.reader[-1])

    def _tf(self, img):
        from PIL import Image
        img = img.convert("RGB")
        w, h = img.size
        s = self.size / min(w, h)
        nw, nh = max(self.size, round(w * s)), max(self.size, round(h * s))
        img = img.resize((nw, nh), Image.BILINEAR, reducing_gap=3.0)
        l, t = (nw - self.size) // 2, (nh - self.size) // 2
        return np.asarray(img.crop((l, t, l + self.size, t + self.size)), dtype=np.uint8)

    def __getitem__(self, i):
        from PIL import Image
        Image.MAX_IMAGE_PIXELS = None
        kind = self.reader[0]
        if kind == "files":
            img = Image.open(self.reader[1][i])
            try:
                img.draft("RGB", (2 * self.size, 2 * self.size))   # JPEG only; no-op otherwise
            except Exception:
                pass
            return self._tf(img)
        if kind == "h5":
            import h5py
            if self._h5 is None:
                self._h5 = h5py.File(self.reader[1], "r")["x"]
            return self._tf(Image.fromarray(np.asarray(self._h5[self.reader[2][i]])))
        if kind == "torchvision":
            if self._tv is None:
                self._tv, _ = _tv(self.reader[1])
            img, _ = self._tv[self.reader[2][i]]
            return self._tf(img)
        raise ValueError(kind)


def _sources_present(man, n_check=25):
    kind = man["reader"][0]
    if kind == "files":
        paths = man["reader"][1]
        step = max(1, len(paths) // n_check)
        return all(os.path.exists(p) for p in paths[::step]) and os.path.exists(paths[-1])
    if kind == "h5":
        return os.path.exists(man["reader"][1])
    return True          # torchvision: checked by ensure_sources itself


def ensure_sources(name, man):
    """Make sure the raw images behind a saved manifest exist in THIS session.

    Manifests live on Drive and survive a Colab restart; downloaded images live
    in /content and do not. Without this, a resumed run tries to decode files
    that are gone (the CRC-VAL-HE-7K FileNotFoundError). Re-running the builder
    re-downloads the data; the rebuilt manifest must hash-match the saved one,
    so the row order cannot silently change between sessions."""
    kind = man["reader"][0]
    if kind == "torchvision":
        # download ONCE in the main process; otherwise every DataLoader worker
        # would start its own download of the same archive
        _tv(man["reader"][1])
        return
    if _sources_present(man):
        return
    if name in ("bracs", "sicapv2", "mhist"):
        raise FileNotFoundError(f"[{name}] source images missing on Drive: e.g. "
                                f"{man['reader'][1][0]} -- check the PATHS block in config.py")
    print(f"  [{name}] source images not in this session (runtime was reset); "
          f"downloading again ...", flush=True)
    m = BUILDERS[name]()
    labels = np.asarray(m["labels"], int)
    groups = None if m["groups"] is None else np.asarray(m["groups"])
    keep = stratified_cap(labels, dataset_cfg(name)["cap"], groups=groups)
    uids = [m["uids"][i] for i in keep]
    h = manifest_hash(uids, labels[keep].tolist())
    if h != man["hash"]:
        raise RuntimeError(f"[{name}] re-downloaded data does not reproduce the saved manifest "
                           f"(hash {h[:12]} vs {man['hash'][:12]}); refusing to mix row orders")
    src = list(m["reader"])
    src[-1] = [src[-1][i] for i in keep]
    if src[-1] != man["reader"][-1]:
        raise RuntimeError(f"[{name}] source paths changed after re-download")
    if m.get("zip"):
        extract_needed(m["zip"], src[-1])
    if not _sources_present(man):
        raise FileNotFoundError(f"[{name}] sources still missing after re-download")
    print(f"  [{name}] sources restored; manifest hash verified", flush=True)


def _cache_paths(name):
    return (os.path.join(CFG.LOCAL_IMG, f"{name}.npy"),
            os.path.join(CFG.LOCAL_IMG, f"{name}.progress.json"),
            os.path.join(CFG.IMG_DRIVE_DIR, f"{name}.npy"))


def _is_complete(npy, man):
    meta = npy + ".meta.json"
    if not (os.path.exists(npy) and os.path.exists(meta)):
        return False
    with open(meta) as fh:
        m = json.load(fh)
    return m.get("hash") == man["hash"] and m.get("complete") is True


def ensure_images(name, workers=8, chunk=1024):
    """Return a read-only memmap (n, 224, 224, 3) uint8 for dataset `name`.

    Order of preference: complete local cache -> complete Drive cache (copied
    to local) -> decode now (resumable chunk by chunk)."""
    import torch
    from torch.utils.data import DataLoader
    _mk(CFG.LOCAL_IMG, CFG.IMG_DRIVE_DIR)
    man = build_manifest(name)
    local, prog, drive = _cache_paths(name)
    if _is_complete(local, man):
        return np.load(local, mmap_mode="r"), man
    if _is_complete(drive, man):
        print(f"  [{name}] copying image cache from Drive ...")
        shutil.copy(drive, local)
        shutil.copy(drive + ".meta.json", local + ".meta.json")
        return np.load(local, mmap_mode="r"), man

    ensure_sources(name, man)
    n, S = man["n"], CFG.IMAGE_SIZE
    if os.path.exists(local) and os.path.exists(prog):
        arr = np.load(local, mmap_mode="r+")
        with open(prog) as fh:
            done = set(json.load(fh)["done"])
    else:
        arr = np.lib.format.open_memmap(local, mode="w+", dtype=np.uint8, shape=(n, S, S, 3))
        done = set()
    chunks = [(s, min(s + chunk, n)) for s in range(0, n, chunk)]
    todo = [i for i in range(len(chunks)) if i not in done]
    print(f"  [{name}] decoding {n} images: {len(todo)}/{len(chunks)} chunks to go")
    dec = _Decoder(man)
    t0 = time.time()
    for k, ci in enumerate(todo):
        a, b = chunks[ci]
        sub = torch.utils.data.Subset(dec, list(range(a, b)))
        dl = DataLoader(sub, batch_size=64, num_workers=workers, shuffle=False,
                        collate_fn=lambda xs: np.stack(xs))
        pos = a
        for batch in dl:
            arr[pos:pos + len(batch)] = batch
            pos += len(batch)
        assert pos == b
        arr.flush()
        done.add(ci)
        with open(prog, "w") as fh:
            json.dump({"done": sorted(done)}, fh)
        el = time.time() - t0
        eta = el / (k + 1) * (len(todo) - k - 1) / 60
        print(f"    chunk {ci + 1}/{len(chunks)}  {b}/{n}  ETA {eta:5.1f} min", flush=True)
    del arr
    # sanity: no all-zero images (an undecoded row would be all zeros)
    arr = np.load(local, mmap_mode="r")
    zero = [i for i in range(0, n, max(1, n // 500)) if not arr[i].any()]
    if zero:
        raise RuntimeError(f"{len(zero)} sampled rows are all-zero, e.g. {zero[:5]}")
    with open(local + ".meta.json", "w") as fh:
        json.dump({"hash": man["hash"], "complete": True, "n": n}, fh)
    if os.path.exists(prog):
        os.remove(prog)
    if dataset_cfg(name).get("persist") or (PERSIST_DOWNLOADED and name in DOWNLOADED):
        print(f"  [{name}] persisting image cache to Drive ({arr.nbytes / 1e9:.2f} GB) ...")
        shutil.copy(local, drive + ".tmp")
        os.replace(drive + ".tmp", drive)
        shutil.copy(local + ".meta.json", drive + ".meta.json")
    return arr, man


def drop_local_images(name):
    local, prog, _ = _cache_paths(name)
    for p in (local, local + ".meta.json", prog):
        if os.path.exists(p):
            os.remove(p)

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/encoders.py
"""Encoder construction and frozen-feature extraction.

Every encoder is wrapped so that it takes a uint8 batch (B, 224, 224, 3) and
returns float32 features (B, d). Each wrapper records what it actually did
(input size, mean/std, pooling, parameter count, weight checksum) so the paper
can state it and a reviewer can check it.

Load safety. The v1 CTransPath failure was a checkpoint silently loaded into the
wrong module. For hub models we rely on the library's strict load (timm and
transformers both raise on missing weights unless told otherwise) and we add a
second guard after extraction: feature statistics (effective rank, dead
dimensions, finiteness) and a quick probe accuracy per encoder are tabulated so
an encoder that "works" but is degenerate stands out. CTransPath, whose
checkpoint we load ourselves, runs in a separate process with the original
timm 0.5.4 and an explicit full-load assertion (ctranspath_worker.py).
"""

from __future__ import annotations

import hashlib
import os

import numpy as np

import config as CFG

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def hf_login():
    """Use HF_TOKEN from Colab secrets (key icon in the left bar) if present."""
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:
            tok = None
    if tok:
        os.environ["HF_TOKEN"] = tok
        try:
            from huggingface_hub import login
            login(token=tok, add_to_git_credential=False)
        except Exception as e:
            print("  HF login failed:", e)
        return True
    print("  No HF_TOKEN found: gated encoders (UNI, UNI2-h, Virchow2, H-optimus-0, "
          "Prov-GigaPath, Hibou-B) will be skipped.")
    return False


def _timm_kwargs(spec):
    import torch
    kw = spec.get("kw") or {}
    if kw == "UNI2_H":
        import timm
        return dict(img_size=224, patch_size=14, depth=24, num_heads=24, init_values=1e-5,
                    embed_dim=1536, mlp_ratio=2.66667 * 2, num_classes=0, no_embed_class=True,
                    mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU,
                    reg_tokens=8, dynamic_img_size=True)
    if kw == "SWIGLU":
        import timm
        return dict(mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU)
    return dict(kw)


def _shim_transformers_onnx():
    """transformers >= 5 removed `transformers.onnx`. Hibou-B's remote
    configuration file still imports `OnnxConfig` from it, only to define an
    ONNX-export helper class that feature extraction never uses. Register a
    stand-in module so that import succeeds. Does nothing if the real module
    exists. This changes no weights and no forward computation."""
    import importlib
    import sys
    import types
    try:
        importlib.import_module("transformers.onnx")
        return
    except Exception:
        pass
    mod = types.ModuleType("transformers.onnx")

    class OnnxConfig:                                   # never instantiated for inference
        default_onnx_opset = 14

        def __init__(self, *a, **k):
            raise RuntimeError("ONNX export is not available (stand-in module)")

    mod.OnnxConfig = OnnxConfig
    mod.OnnxConfigWithPast = OnnxConfig
    mod.OnnxSeq2SeqConfigWithPast = OnnxConfig
    sys.modules["transformers.onnx"] = mod
    try:
        import transformers
        transformers.onnx = mod
    except Exception:
        pass


class Encoder:
    def __init__(self, name, spec, device):
        self.name, self.spec, self.device = name, spec, device
        self.kind = spec["kind"]
        self.info = {"name": name, "kind": self.kind, "id": spec.get("id")}
        if self.kind == "timm":
            self._build_timm()
        elif self.kind == "hf":
            self._build_hf()
        else:
            raise ValueError(f"{self.kind} is not built in-process")
        self.model.eval().to(device)
        p = sum(t.numel() for t in self.model.parameters())
        self.info["n_params"] = int(p)
        self.info["weight_checksum"] = self._checksum()

    # -------------------------------------------------------------- builders
    def _build_timm(self):
        import timm
        kw = _timm_kwargs(self.spec)
        kw.setdefault("num_classes", 0)
        m = timm.create_model(self.spec["id"], pretrained=True, **kw)
        cfg = timm.data.resolve_model_data_config(m)
        native = tuple(cfg.get("input_size", (3, 224, 224)))[-1]
        if native != CFG.IMAGE_SIZE and "dynamic_img_size" not in kw and "img_size" not in kw:
            # e.g. DINOv2 defaults to 518. Rebuild at 224; timm resamples the
            # position embedding from the checkpoint. Recorded in self.info.
            # CNNs (ResNet, ConvNeXt) take any input size and reject these ViT-only
            # arguments; they are simply run at 224 as built.
            try:
                m = timm.create_model(self.spec["id"], pretrained=True,
                                      img_size=CFG.IMAGE_SIZE, **kw)
                self.info["resized_from"] = native
            except TypeError:
                try:
                    m = timm.create_model(self.spec["id"], pretrained=True,
                                          dynamic_img_size=True, **kw)
                    self.info["dynamic_from"] = native
                except TypeError:
                    self.info["run_at_224_native"] = native
        self.model = m
        self.mean = tuple(self.spec.get("mean") or cfg.get("mean") or IMAGENET_MEAN)
        self.std = tuple(self.spec.get("std") or cfg.get("std") or IMAGENET_STD)
        self.pool = self.spec.get("pool", "default")
        self.info.update(mean=self.mean, std=self.std, pool=self.pool,
                         timm_version=timm.__version__,
                         num_prefix_tokens=int(getattr(m, "num_prefix_tokens", 0) or 0))

    def _build_hf(self):
        import transformers
        from transformers import AutoImageProcessor, AutoModel
        trc = bool(self.spec.get("trust_remote_code", False))
        if trc:
            _shim_transformers_onnx()
        self.model = AutoModel.from_pretrained(self.spec["id"], trust_remote_code=trc)
        try:
            proc = AutoImageProcessor.from_pretrained(self.spec["id"], trust_remote_code=trc)
            self.mean = tuple(proc.image_mean)
            self.std = tuple(proc.image_std)
        except Exception:
            self.mean, self.std = IMAGENET_MEAN, IMAGENET_STD
        self.pool = self.spec.get("pool", "cls")
        self.info.update(mean=self.mean, std=self.std, pool=self.pool,
                         transformers_version=transformers.__version__)

    def _checksum(self):
        h = hashlib.sha256()
        for i, t in enumerate(self.model.state_dict().values()):
            if i % 7 == 0 and t.numel():
                h.update(np.ascontiguousarray(t.detach().float().flatten()[:64].cpu().numpy()).tobytes())
        return h.hexdigest()[:16]

    # -------------------------------------------------------------- forward
    def __call__(self, x_uint8):
        """x_uint8: torch uint8 tensor (B, H, W, 3) on any device."""
        import torch
        x = x_uint8.to(self.device, non_blocking=True).permute(0, 3, 1, 2).float().div_(255.0)
        mean = torch.tensor(self.mean, device=self.device).view(1, 3, 1, 1)
        std = torch.tensor(self.std, device=self.device).view(1, 3, 1, 1)
        x = (x - mean) / std
        use_amp = self.device.startswith("cuda")
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            if self.kind == "timm":
                if self.pool == "cls+meanpatch":
                    tok = self.model.forward_features(x)
                    npt = self.info["num_prefix_tokens"] or 1
                    f = torch.cat([tok[:, 0], tok[:, npt:].mean(1)], dim=-1)
                else:
                    f = self.model(x)
                    if f.ndim == 3:
                        f = f.mean(1)
                    elif f.ndim == 4:
                        f = f.mean((2, 3))
            else:
                out = self.model(pixel_values=x)
                if self.pool == "pooler" and getattr(out, "pooler_output", None) is not None:
                    f = out.pooler_output
                else:
                    f = out.last_hidden_state[:, 0]
        return f.float().cpu().numpy()


def build(name, spec, device):
    return Encoder(name, spec, device)


# ---------------------------------------------------------------------------
# guards
# ---------------------------------------------------------------------------

def feature_stats(F):
    F = np.asarray(F, np.float64)
    Fc = F - F.mean(0, keepdims=True)
    s = np.linalg.svd(Fc[: min(len(Fc), 4000)], compute_uv=False)
    p = s ** 2 / max((s ** 2).sum(), 1e-12)
    eff_rank = float(np.exp(-(p * np.log(p + 1e-12)).sum()))
    return {"n": int(F.shape[0]), "dim": int(F.shape[1]),
            "finite": bool(np.isfinite(F).all()),
            "mean_norm": float(np.linalg.norm(F, axis=1).mean()),
            "eff_rank": eff_rank, "eff_rank_frac": eff_rank / F.shape[1],
            "dead_dims_frac": float((F.std(0) < 1e-8).mean())}

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/extract.py
"""Resumable feature extraction.

Layout on Drive:
    features/<dataset>/<encoder>.npy        (n, d) float32, rows in manifest order
    features/<dataset>/<encoder>.json       provenance: encoder info, manifest hash, stats
    features/<dataset>/_shards_<encoder>/   temporary; deleted after assembly

A Colab disconnect loses at most one shard (SHARD_ROWS rows) of one encoder.
Re-running the notebook cell resumes exactly where it stopped.
"""

from __future__ import annotations

import glob
import json
import os
import shutil
import subprocess
import sys
import time

import numpy as np

import config as CFG
import encoders as E

TIMM054_DIR = os.environ.get("V3_TIMM054", "/content/timm054")


def _paths(ds, enc):
    d = os.path.join(CFG.FEAT_DIR, ds)
    return d, os.path.join(d, f"{enc}.npy"), os.path.join(d, f"{enc}.json"), \
        os.path.join(d, f"_shards_{enc}")


def is_done(ds, enc, man):
    _, npy, meta, _ = _paths(ds, enc)
    if not (os.path.exists(npy) and os.path.exists(meta)):
        return False
    with open(meta) as fh:
        m = json.load(fh)
    return m.get("manifest_hash") == man["hash"] and m.get("complete") is True


def _shard_ranges(n):
    return [(s, min(s + CFG.SHARD_ROWS, n)) for s in range(0, n, CFG.SHARD_ROWS)]


def _assemble(ds, enc, man, info, t_start):
    d, npy, meta, sh = _paths(ds, enc)
    ranges = _shard_ranges(man["n"])
    parts = [np.load(os.path.join(sh, f"shard_{i:04d}.npy")) for i in range(len(ranges))]
    F = np.concatenate(parts).astype(CFG.FEATURE_DTYPE)
    assert F.shape[0] == man["n"], f"{F.shape[0]} rows != manifest {man['n']}"
    stats = E.feature_stats(F)
    if not stats["finite"]:
        raise RuntimeError(f"{ds}/{enc}: non-finite features; not saving")
    np.save(npy + ".tmp.npy", F)
    os.replace(npy + ".tmp.npy", npy)
    with open(meta, "w") as fh:
        json.dump({"dataset": ds, "encoder": enc, "manifest_hash": man["hash"],
                   "shape": list(F.shape), "dtype": CFG.FEATURE_DTYPE, "complete": True,
                   "encoder_info": info, "stats": stats,
                   "minutes": round((time.time() - t_start) / 60, 1),
                   "saved": time.strftime("%Y-%m-%d %H:%M:%S")}, fh, indent=1, default=str)
    shutil.rmtree(sh, ignore_errors=True)
    return stats


def extract_inprocess(ds, enc_name, spec, images, man, device, enc_obj=None):
    import torch
    d, npy, meta, sh = _paths(ds, enc_name)
    t0 = time.time()
    enc = enc_obj or E.build(enc_name, spec, device)   # build first: a failed load leaves no trace
    os.makedirs(sh, exist_ok=True)
    ranges = _shard_ranges(man["n"])
    todo = [i for i in range(len(ranges))
            if not os.path.exists(os.path.join(sh, f"shard_{i:04d}.npy"))]
    print(f"    {enc_name}: {enc.info.get('n_params', 0) / 1e6:.0f}M params, "
          f"{len(todo)}/{len(ranges)} shards to go", flush=True)
    for k, i in enumerate(todo):
        a, b = ranges[i]
        out = []
        for s in range(a, b, CFG.EXTRACT_BATCH):
            x = torch.from_numpy(np.ascontiguousarray(images[s:min(s + CFG.EXTRACT_BATCH, b)]))
            out.append(enc(x))
        np.save(os.path.join(sh, f"shard_{i:04d}.npy"), np.concatenate(out).astype(np.float32))
        el = time.time() - t0
        print(f"      shard {i + 1}/{len(ranges)}  {b}/{man['n']}  "
              f"{(b - a) * (k + 1) / max(el, 1e-9):6.0f} img/s  "
              f"ETA {el / (k + 1) * (len(todo) - k - 1) / 60:5.1f} min", flush=True)
    info = dict(enc.info)
    stats = _assemble(ds, enc_name, man, info, t0)
    return stats


# ---------------------------------------------------------------------------
# CTransPath: separate process with the original timm 0.5.4
# ---------------------------------------------------------------------------

def ensure_timm054():
    if os.path.exists(os.path.join(TIMM054_DIR, "timm", "__init__.py")):
        return
    print("  installing timm==0.5.4 into an isolated directory (for CTransPath only) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "--target", TIMM054_DIR, "timm==0.5.4"], check=True)


def _run_streaming(cmd, env):
    """Run a worker and echo its output line by line INTO THE NOTEBOOK.
    (Colab does not display a child process's own stdout/stderr.)"""
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        line = line.rstrip("\n")
        tail = (tail + [line])[-40:]
        if ("Warning" in line or "warn(" in line) and "Error" not in line:
            continue
        print(line, flush=True)
    p.wait()
    return p.returncode, "\n".join(tail)


# Hugging Face models whose remote code needs an older transformers than Colab's.
# They run in a separate process against an isolated install (like CTransPath).
ISOLATED_HF = {"hibou_b": "4.46.3"}
OLD_TF_DIR = os.environ.get("V3_OLD_TF", "/content/transformers_4_46")


def ensure_old_transformers(version="4.46.3"):
    marker = os.path.join(OLD_TF_DIR, f".installed_{version}")
    if os.path.exists(marker):
        return
    print(f"  installing transformers=={version} into an isolated directory "
          f"(for remote-code models only) ...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "--target", OLD_TF_DIR, f"transformers=={version}",
                    "tokenizers==0.20.3", "huggingface_hub==0.26.5"], check=True)
    open(marker, "w").close()


def extract_hf_isolated(ds, enc_name, spec, images_path, man, device):
    d, npy, meta, sh = _paths(ds, enc_name)
    os.makedirs(sh, exist_ok=True)
    ensure_old_transformers(ISOLATED_HF[enc_name])
    t0 = time.time()
    env = dict(os.environ)
    env["PYTHONPATH"] = OLD_TF_DIR + os.pathsep + CFG.CODE_DIR
    env["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
    cmd = [sys.executable, os.path.join(CFG.CODE_DIR, "hf_worker.py"), "--model", spec["id"],
           "--pool", spec.get("pool", "cls"), "--images", images_path, "--shards", sh,
           "--shard_rows", str(CFG.SHARD_ROWS), "--batch", str(CFG.EXTRACT_BATCH),
           "--device", device]
    if spec.get("trust_remote_code"):
        cmd.append("--trust_remote_code")
    rc, tail = _run_streaming(cmd, env)
    if rc != 0:
        if not any(f.startswith("shard_") for f in os.listdir(sh)):
            shutil.rmtree(sh, ignore_errors=True)        # nothing done: leave no "partial"
        raise RuntimeError(f"{enc_name} worker failed: {tail.splitlines()[-1] if tail else rc}")
    with open(os.path.join(sh, "load_report.json")) as fh:
        rep = json.load(fh)
    info = {"name": enc_name, "kind": "hf-isolated", "id": spec["id"],
            "mean": rep["mean"], "std": rep["std"], "pool": rep["pool"],
            "n_params": rep["n_params"], "load_report": rep}
    return _assemble(ds, enc_name, man, info, t0)


def extract_ctranspath(ds, images_path, man, device):
    d, npy, meta, sh = _paths(ds, "ctranspath")
    os.makedirs(sh, exist_ok=True)
    if not os.path.exists(CFG.CTRANSPATH_CKPT):
        raise FileNotFoundError(CFG.CTRANSPATH_CKPT)
    ensure_timm054()
    t0 = time.time()
    env = dict(os.environ)
    env["PYTHONPATH"] = TIMM054_DIR + os.pathsep + CFG.CODE_DIR
    worker = os.path.join(CFG.CODE_DIR, "ctranspath_worker.py")
    cmd = [sys.executable, worker, "--images", images_path, "--ckpt", CFG.CTRANSPATH_CKPT,
           "--shards", sh, "--shard_rows", str(CFG.SHARD_ROWS),
           "--batch", str(CFG.EXTRACT_BATCH), "--device", device]
    rc, _ = _run_streaming(cmd, env)
    if rc != 0:
        raise RuntimeError("CTransPath worker failed (see output above); nothing saved")
    with open(os.path.join(sh, "load_report.json")) as fh:
        rep = json.load(fh)
    info = {"name": "ctranspath", "kind": "ctranspath", "id": CFG.CTRANSPATH_CKPT,
            "mean": E.IMAGENET_MEAN, "std": E.IMAGENET_STD, "pool": "avg (timm 0.5.4 head)",
            "load_report": rep}
    return _assemble(ds, "ctranspath", man, info, t0)


# ---------------------------------------------------------------------------
# driver
# ---------------------------------------------------------------------------

def run_domain(datasets, encoders, device="cuda", only=None, skip_encoders=()):
    """Extract every (dataset, encoder) that is not already complete."""
    import imgcache as IC
    log = []
    E.hf_login()
    for ds in datasets:
        if only and ds not in only:
            continue
        pending = None
        print(f"\n=== {ds} ===", flush=True)
        try:
            man = IC.build_manifest(ds)
        except Exception as e:
            print(f"  !! manifest failed: {type(e).__name__}: {e}")
            log.append({"dataset": ds, "encoder": None, "status": f"manifest failed: {e}"})
            continue
        pending = [e for e in encoders if e not in skip_encoders and not is_done(ds, e, man)]
        if not pending:
            print("  all encoders complete")
            continue
        images, man = IC.ensure_images(ds)
        images_path = IC._cache_paths(ds)[0]
        for enc_name in pending:
            spec = encoders[enc_name]
            t = time.time()
            try:
                if spec["kind"] == "ctranspath":
                    stats = extract_ctranspath(ds, images_path, man, device)
                elif enc_name in ISOLATED_HF:
                    stats = extract_hf_isolated(ds, enc_name, spec, images_path, man, device)
                else:
                    stats = extract_inprocess(ds, enc_name, spec, images, man, device)
                msg = (f"ok  dim={stats['dim']}  eff_rank={stats['eff_rank']:.0f} "
                       f"({stats['eff_rank_frac']:.0%})  {(time.time() - t) / 60:.1f} min")
            except Exception as e:
                msg = f"SKIPPED: {type(e).__name__}: {str(e)[:300]}"
            print(f"  -> {enc_name}: {msg}", flush=True)
            log.append({"dataset": ds, "encoder": enc_name, "status": msg})
            try:
                import torch
                torch.cuda.empty_cache()
            except Exception:
                pass
    os.makedirs(CFG.LOG_DIR, exist_ok=True)
    with open(os.path.join(CFG.LOG_DIR, f"extract_{time.strftime('%Y%m%d_%H%M%S')}.json"), "w") as fh:
        json.dump(log, fh, indent=1)
    return log


def status_table(datasets, encoders):
    """Which (dataset, encoder) features exist and are complete."""
    import imgcache as IC
    rows = []
    for ds in datasets:
        jp = os.path.join(CFG.MANIFEST_DIR, f"{ds}.json")
        man = json.load(open(jp)) if os.path.exists(jp) else None
        row = {"dataset": ds, "n": man["n"] if man else None}
        for e in encoders:
            row[e] = ("ok" if man and is_done(ds, e, man) else
                      ("partial" if os.path.isdir(_paths(ds, e)[3]) else "-"))
        rows.append(row)
    return rows


# ---------------------------------------------------------------------------
# consistency with the v2 features (pathology only)
# ---------------------------------------------------------------------------

V2_PREFERRED = ("trainval_features.npy", "all_features.npy", "features.npy")


def _find_v2(ds, enc):
    """Locate the v2 feature file covering the SAME rows as the v3 manifest.

    v3 fix: the previous version took the alphabetically first *features*.npy,
    which on BRACS was test_features.npy (the 570-image held-out institutions),
    so the comparison reported "row count differs" for UNI, Phikon-v2 and
    ResNet-50. Split-specific files (test_/train_/val_) are now never used."""
    if enc == "ctranspath":
        d = CFG.V2_CTRANSPATH_FIXED[ds]
    else:
        d = os.path.join(CFG.V2_CACHE[ds], CFG.V2_NAME[enc])
    if not os.path.isdir(d):
        return None, None
    present = set(os.listdir(d))
    fp = next((os.path.join(d, f) for f in V2_PREFERRED if f in present), None)
    if fp is None:
        cands = sorted(f for f in present if f.endswith(".npy") and "features" in f
                       and not f.startswith(("test_", "train_", "val_")))
        fp = os.path.join(d, cands[0]) if cands else None
    if fp is None:
        return None, None
    lp = os.path.join(d, os.path.basename(fp).replace("features", "labels"))
    return fp, (lp if os.path.exists(lp) else None)


def compare_with_v2(ds, enc, n_rows=3000):
    """Linear CKA between v3 and v2 features on the same rows.

    ~1.0 means the v3 pipeline reproduces v2's features for this encoder (the
    only differences are decode details). A low value means row order or
    preprocessing differs, which must be explained before using either set."""
    from cka import linear_cka
    if ds not in CFG.V2_CACHE or enc not in CFG.V2_NAME:
        return None
    fp, lp = _find_v2(ds, enc)
    _, npy, _, _ = _paths(ds, enc)
    if fp is None or not os.path.exists(npy):
        return {"dataset": ds, "encoder": enc, "status": "missing v2 or v3 features"}
    old = np.load(fp, mmap_mode="r")
    new = np.load(npy, mmap_mode="r")
    man = json.load(open(os.path.join(CFG.MANIFEST_DIR, f"{ds}.json")))
    res = {"dataset": ds, "encoder": enc, "v2_file": os.path.relpath(fp, CFG.DRIVE),
           "n_v2": int(old.shape[0]), "n_v3": int(new.shape[0])}
    if old.shape[0] != new.shape[0]:
        res["status"] = "row count differs"
        return res
    if lp:
        y_old = np.load(lp)
        res["labels_identical"] = bool(np.array_equal(y_old, np.asarray(man["labels"])))
    idx = np.sort(np.random.default_rng(0).choice(new.shape[0], min(n_rows, new.shape[0]),
                                                  replace=False))
    res["cka_v2_v3"] = float(linear_cka(np.asarray(old[idx], np.float64),
                                        np.asarray(new[idx], np.float64)))
    res["status"] = ("reproduces v2" if res["cka_v2_v3"] > 0.98 else
                     "close to v2" if res["cka_v2_v3"] > 0.90 else
                     "DIFFERS FROM v2 - investigate before analysis")
    return res


# ---------------------------------------------------------------------------
# ResNet-50 provenance: which variant did v2's "resnet50_imagenet" use?
# Diagnostic only -- it does not change the registered v3 encoder.
# ---------------------------------------------------------------------------

RESNET_CANDIDATES = {
    "torchvision IMAGENET1K_V1 (v3)": ("timm", "resnet50.tv_in1k"),
    "torchvision IMAGENET1K_V2":      ("timm", "resnet50.tv2_in1k"),
    "timm a1_in1k":                   ("timm", "resnet50.a1_in1k"),
    "timm fb_swsl_ig1b_ft_in1k":      ("timm", "resnet50.fb_swsl_ig1b_ft_in1k"),
}


def resnet_provenance(ds="mhist", n_rows=1500, device="cuda"):
    """Compare v2's ResNet-50 features with several ImageNet ResNet-50 variants
    under two preprocessings, on the same rows.

    preprocessing "v3": the shared 224 cache (Resize 224 + CenterCrop 224)
    preprocessing "tv": torchvision's classic eval transform (Resize 256 +
                        CenterCrop 224) applied to the original images
    The row with CKA ~1.0 identifies what v2 used."""
    import torch
    from PIL import Image
    from cka import linear_cka
    import imgcache as IC
    fp, _ = _find_v2(ds, "resnet50_in1k")
    if fp is None:
        raise FileNotFoundError(f"no v2 ResNet-50 features for {ds}")
    old = np.load(fp, mmap_mode="r")
    images, man = IC.ensure_images(ds)
    idx = np.sort(np.random.default_rng(0).choice(man["n"], min(n_rows, man["n"]), replace=False))
    ref = np.asarray(old[idx], np.float64)

    def tv_batch(rows):
        out = []
        for i in rows:
            img = Image.open(man["reader"][1][i]).convert("RGB")
            w, h = img.size
            s = 256 / min(w, h)
            img = img.resize((max(256, round(w * s)), max(256, round(h * s))), Image.BILINEAR)
            W, H = img.size
            l, t = (W - 224) // 2, (H - 224) // 2
            out.append(np.asarray(img.crop((l, t, l + 224, t + 224)), np.uint8))
        return np.stack(out)

    can_tv = man["reader"][0] == "files"
    rows = []
    for label, (kind, mid) in RESNET_CANDIDATES.items():
        try:
            enc = E.build(label, dict(kind=kind, id=mid), device)
        except Exception as e:
            rows.append({"variant": label, "preprocessing": "-", "cka_with_v2": None,
                         "note": f"load failed: {type(e).__name__}"})
            continue
        for prep in (("v3", "tv") if can_tv else ("v3",)):
            feats = []
            for s in range(0, len(idx), 64):
                b = idx[s:s + 64]
                x = np.asarray(images[b]) if prep == "v3" else tv_batch(b)
                feats.append(enc(torch.from_numpy(np.ascontiguousarray(x))))
            Fn = np.concatenate(feats).astype(np.float64)
            rows.append({"variant": label, "preprocessing":
                         "Resize224+Crop224 (v3)" if prep == "v3" else "Resize256+Crop224",
                         "cka_with_v2": float(linear_cka(ref, Fn)), "note": ""})
        del enc
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return rows

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/ctranspath_worker.py
"""CTransPath feature extraction, run as a SEPARATE PROCESS under timm 0.5.4.

Why a separate process: CTransPath's released checkpoint matches the Swin
implementation of timm 0.5.4, which is the setup v2 verified (all 200 entries,
missing=0, unexpected=0, 16 ConvStem tensors). The other encoders need a modern
timm. Two timm versions cannot coexist in one Python process, so this script
runs with PYTHONPATH pointing at an isolated timm 0.5.4 install.

Hard guard: the script refuses to extract unless the checkpoint load reports
zero missing and zero unexpected keys AND the ConvStem tensors were loaded.
That is the exact check whose absence produced the v1 bug.

Usage (called by extract.extract_ctranspath):
    python ctranspath_worker.py --images X.npy --ckpt ctranspath.pth --shards DIR
"""

import argparse
import json
import os
import time

import numpy as np
import torch
import torch.nn as nn


class ConvStem(nn.Module):
    """Convolutional patch embedding of CTransPath (Wang et al., 2022)."""

    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=768,
                 norm_layer=None, flatten=True):
        super().__init__()
        assert patch_size == 4 and embed_dim % 8 == 0
        self.img_size = (img_size, img_size)
        self.patch_size = (patch_size, patch_size)
        self.grid_size = (img_size // patch_size, img_size // patch_size)
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.flatten = flatten
        stem, in_dim, out_dim = [], in_chans, embed_dim // 8
        for _ in range(2):
            stem += [nn.Conv2d(in_dim, out_dim, 3, 2, 1, bias=False),
                     nn.BatchNorm2d(out_dim), nn.ReLU(inplace=True)]
            in_dim, out_dim = out_dim, out_dim * 2
        stem.append(nn.Conv2d(in_dim, embed_dim, 1))
        self.proj = nn.Sequential(*stem)
        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()

    def forward(self, x):
        x = self.proj(x)
        if self.flatten:
            x = x.flatten(2).transpose(1, 2)
        return self.norm(x)


def build(ckpt):
    import timm
    if not timm.__version__.startswith("0.5."):
        raise RuntimeError(f"CTransPath worker must run under timm 0.5.x, got {timm.__version__}")
    m = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=0)
    ed = getattr(m, "embed_dim", 96)
    if isinstance(ed, (list, tuple)):
        ed = ed[0]
    m.patch_embed = ConvStem(embed_dim=ed, norm_layer=nn.LayerNorm)
    # weights_only=False: torch>=2.6 defaults to True, which rejects this
    # (trusted, your own) checkpoint format.
    sd = torch.load(ckpt, map_location="cpu", weights_only=False)
    state = sd["model"] if isinstance(sd, dict) and "model" in sd else sd
    res = m.load_state_dict(state, strict=False)
    missing, unexpected = list(res.missing_keys), list(res.unexpected_keys)
    stem_keys = [k for k in state if k.startswith("patch_embed")]
    report = {"timm": timm.__version__, "torch": torch.__version__,
              "ckpt_entries": len(state), "missing": missing, "unexpected": unexpected,
              "patch_embed_tensors_in_ckpt": len(stem_keys)}
    print(f"  [ctranspath] timm {timm.__version__}  entries={len(state)}  "
          f"missing={len(missing)}  unexpected={len(unexpected)}  "
          f"ConvStem tensors={len(stem_keys)}", flush=True)
    if missing or unexpected or not stem_keys:
        raise SystemExit("INCOMPLETE CTransPath LOAD - refusing to extract "
                         f"(missing {missing[:5]}, unexpected {unexpected[:5]})")
    print("  [ctranspath] LOAD VERIFIED", flush=True)
    return m.eval(), report


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--images", required=True)
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--shards", required=True)
    ap.add_argument("--shard_rows", type=int, default=2048)
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--device", default="cuda")
    a = ap.parse_args()
    dev = a.device if (a.device != "cuda" or torch.cuda.is_available()) else "cpu"
    os.makedirs(a.shards, exist_ok=True)
    model, report = build(a.ckpt)
    model.to(dev)
    with open(os.path.join(a.shards, "load_report.json"), "w") as fh:
        json.dump(report, fh, indent=1)
    imgs = np.load(a.images, mmap_mode="r")
    n = imgs.shape[0]
    mean = torch.tensor((0.485, 0.456, 0.406), device=dev).view(1, 3, 1, 1)
    std = torch.tensor((0.229, 0.224, 0.225), device=dev).view(1, 3, 1, 1)
    ranges = [(s, min(s + a.shard_rows, n)) for s in range(0, n, a.shard_rows)]
    todo = [i for i in range(len(ranges))
            if not os.path.exists(os.path.join(a.shards, f"shard_{i:04d}.npy"))]
    print(f"  [ctranspath] {n} images, {len(todo)}/{len(ranges)} shards to go", flush=True)
    t0 = time.time()
    with torch.no_grad():                       # fp32, as in the v2 extraction
        for k, i in enumerate(todo):
            lo, hi = ranges[i]
            out = []
            for s in range(lo, hi, a.batch):
                x = torch.from_numpy(np.ascontiguousarray(imgs[s:min(s + a.batch, hi)]))
                x = x.to(dev).permute(0, 3, 1, 2).float().div_(255.0)
                f = model((x - mean) / std)
                if f.ndim == 3:
                    f = f.mean(1)
                out.append(f.float().cpu().numpy())
            np.save(os.path.join(a.shards, f"shard_{i:04d}.npy"),
                    np.concatenate(out).astype(np.float32))
            el = time.time() - t0
            print(f"      shard {i + 1}/{len(ranges)}  {hi}/{n}  "
                  f"ETA {el / (k + 1) * (len(todo) - k - 1) / 60:5.1f} min", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/hf_worker.py
"""Feature extraction for a Hugging Face model whose remote code needs an OLDER
transformers than Colab ships (Hibou-B: its custom DINOv2 code imports helpers
removed in transformers 5).

Run as a separate process with PYTHONPATH pointing at an isolated install of
transformers 4.46 (see extract.ensure_old_transformers). torch, numpy and the
GPU come from the normal Colab environment. Output format is identical to the
in-process path: shard_XXXX.npy files plus a load report.

Usage (called by extract.extract_hf_isolated):
    python hf_worker.py --model histai/hibou-b --pool pooler --images X.npy --shards DIR
"""

import argparse
import json
import os
import time

import numpy as np
import torch


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--pool", default="pooler", choices=["pooler", "cls"])
    ap.add_argument("--images", required=True)
    ap.add_argument("--shards", required=True)
    ap.add_argument("--shard_rows", type=int, default=2048)
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--trust_remote_code", action="store_true")
    a = ap.parse_args()

    import transformers
    from transformers import AutoImageProcessor, AutoModel
    dev = a.device if (a.device != "cuda" or torch.cuda.is_available()) else "cpu"
    os.makedirs(a.shards, exist_ok=True)

    model = AutoModel.from_pretrained(a.model, trust_remote_code=a.trust_remote_code)
    model.eval().to(dev)
    try:
        proc = AutoImageProcessor.from_pretrained(a.model, trust_remote_code=a.trust_remote_code)
        mean, std = list(proc.image_mean), list(proc.image_std)
    except Exception:
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    n_params = int(sum(p.numel() for p in model.parameters()))
    report = {"transformers": transformers.__version__, "torch": torch.__version__,
              "model": a.model, "class": type(model).__name__, "n_params": n_params,
              "mean": mean, "std": std, "pool": a.pool}
    print(f"  [{a.model}] transformers {transformers.__version__}  {type(model).__name__}  "
          f"{n_params / 1e6:.0f}M params  mean={mean} std={std}", flush=True)
    with open(os.path.join(a.shards, "load_report.json"), "w") as fh:
        json.dump(report, fh, indent=1)

    imgs = np.load(a.images, mmap_mode="r")
    n = imgs.shape[0]
    m_t = torch.tensor(mean, device=dev).view(1, 3, 1, 1)
    s_t = torch.tensor(std, device=dev).view(1, 3, 1, 1)
    ranges = [(s, min(s + a.shard_rows, n)) for s in range(0, n, a.shard_rows)]
    todo = [i for i in range(len(ranges))
            if not os.path.exists(os.path.join(a.shards, f"shard_{i:04d}.npy"))]
    print(f"  [{a.model}] {n} images, {len(todo)}/{len(ranges)} shards to go", flush=True)
    t0 = time.time()
    use_amp = dev.startswith("cuda")
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
        for k, i in enumerate(todo):
            lo, hi = ranges[i]
            out = []
            for s in range(lo, hi, a.batch):
                x = torch.from_numpy(np.ascontiguousarray(imgs[s:min(s + a.batch, hi)]))
                x = x.to(dev).permute(0, 3, 1, 2).float().div_(255.0)
                o = model(pixel_values=(x - m_t) / s_t)
                if a.pool == "pooler" and getattr(o, "pooler_output", None) is not None:
                    f = o.pooler_output
                else:
                    f = o.last_hidden_state[:, 0]
                out.append(f.float().cpu().numpy())
            np.save(os.path.join(a.shards, f"shard_{i:04d}.npy"),
                    np.concatenate(out).astype(np.float32))
            el = time.time() - t0
            print(f"      shard {i + 1}/{len(ranges)}  {hi}/{n}  "
                  f"ETA {el / (k + 1) * (len(todo) - k - 1) / 60:5.1f} min", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/text_arm.py
"""Text arm: frozen sentence encoders on public classification sets.

Same downstream pipeline as the image arms (the analysis only ever sees an
(n, d) feature matrix and labels), so this adds independent pools in a
different modality at almost no engineering cost.
"""

from __future__ import annotations

import json
import os
import time

import numpy as np

import config as CFG
import imgcache as IC          # manifest_hash, stratified_cap
import extract as X            # _paths, is_done


def build_text_manifest(name, force=False):
    os.makedirs(CFG.MANIFEST_DIR, exist_ok=True)
    jp = os.path.join(CFG.MANIFEST_DIR, f"{name}.json")
    if os.path.exists(jp) and not force:
        with open(jp) as fh:
            return json.load(fh)
    from datasets import load_dataset
    hf_id, conf, split, tcol, lcol, cap = CFG.TEXT_DATASETS[name]
    ds = load_dataset(hf_id, conf, split=split) if conf else load_dataset(hf_id, split=split)
    texts = [str(t) for t in ds[tcol]]
    raw = list(ds[lcol])
    classes = sorted(set(raw))
    cmap = {c: i for i, c in enumerate(classes)}
    labels = np.array([cmap[v] for v in raw])
    keep_ok = np.array([len(t.strip()) > 0 for t in texts])
    idx_all = np.where(keep_ok)[0]
    sub = IC.stratified_cap(labels[idx_all], cap)
    keep = idx_all[sub]
    man = dict(name=name, n=int(len(keep)), n_before_cap=int(len(labels)), cap=cap,
               seed=CFG.SEED, classes=[str(c) for c in classes],
               uids=[f"{name}_{i}" for i in keep], labels=labels[keep].tolist(),
               groups=None, texts=[texts[i] for i in keep],
               class_counts=np.bincount(labels[keep], minlength=len(classes)).tolist(),
               source=f"{hf_id}:{conf or 'default'}:{split}", n_groups=None,
               built=time.strftime("%Y-%m-%d %H:%M:%S"))
    man["hash"] = IC.manifest_hash(man["uids"], man["labels"])
    with open(jp, "w") as fh:
        json.dump(man, fh)
    print(f"  [{name}] n={man['n']} classes={len(classes)}")
    return man


def run_text(datasets=None, encoders=None, device="cuda", batch=128):
    from sentence_transformers import SentenceTransformer
    import encoders as E
    datasets = datasets or list(CFG.TEXT_DATASETS)
    encoders = encoders or CFG.TEXT_ENCODERS
    mans = {}
    for ds in datasets:
        try:
            mans[ds] = build_text_manifest(ds)
        except Exception as e:
            print(f"  !! {ds}: manifest failed: {type(e).__name__}: {e}")
    log = []
    for enc_name, spec in encoders.items():          # load each model once
        todo = [ds for ds in mans if not X.is_done(ds, enc_name, mans[ds])]
        if not todo:
            print(f"{enc_name}: complete")
            continue
        t0 = time.time()
        try:
            model = SentenceTransformer(spec["id"], device=device)
        except Exception as e:
            print(f"  !! {enc_name}: load failed: {e}")
            continue
        for ds in todo:
            man = mans[ds]
            texts = [spec.get("prefix", "") + t for t in man["texts"]]
            F = model.encode(texts, batch_size=batch, show_progress_bar=False,
                             convert_to_numpy=True, normalize_embeddings=False)
            F = np.asarray(F, dtype=CFG.FEATURE_DTYPE)
            d, npy, meta, _ = X._paths(ds, enc_name)
            os.makedirs(d, exist_ok=True)
            np.save(npy, F)
            stats = E.feature_stats(F)
            with open(meta, "w") as fh:
                json.dump({"dataset": ds, "encoder": enc_name, "manifest_hash": man["hash"],
                           "shape": list(F.shape), "complete": True,
                           "encoder_info": {"name": enc_name, "kind": "st", "id": spec["id"],
                                            "prefix": spec.get("prefix", "")},
                           "stats": stats, "saved": time.strftime("%Y-%m-%d %H:%M:%S")}, fh, indent=1)
            print(f"  {enc_name:14s} {ds:14s} {F.shape}  eff_rank {stats['eff_rank']:.0f}")
            log.append((ds, enc_name, list(F.shape)))
        del model
        print(f"{enc_name}: done in {(time.time() - t0) / 60:.1f} min")
    return log

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/prereg.py
"""Pre-registration of the analysis.

register() writes prereg.json with the PREREG parameters, the dataset/encoder
lists and a SHA-256 of every file that can change a reported number. Upload
that file to OSF (or commit and tag it) BEFORE running notebook 04: the
timestamp from a third party is what makes "pre-registered" checkable.

check() is called by notebook 04. It refuses to proceed if any frozen file or
parameter changed, unless a deviation is declared, in which case the deviation
text is written into every results file.
"""

from __future__ import annotations

import datetime as _dt
import hashlib
import json
import os

import config as CFG

FROZEN_FILES = ["config.py", "cka.py", "folds.py", "fusion.py", "pairs.py", "inference.py",
                "analysis.py", "run_pipeline.py", "analyse_all.py"]


def _sha(path):
    with open(path, "rb") as fh:
        return hashlib.sha256(fh.read()).hexdigest()


def snapshot():
    files = {f: _sha(os.path.join(CFG.CODE_DIR, f)) for f in FROZEN_FILES
             if os.path.exists(os.path.join(CFG.CODE_DIR, f))}
    return {
        "params": CFG.PREREG,
        "datasets": {"pathology": list(CFG.PATHOLOGY_DATASETS),
                     "vision": list(CFG.VISION_DATASETS),
                     "text": list(CFG.TEXT_DATASETS)},
        "encoders": {"pathology": list(CFG.PATHOLOGY_ENCODERS),
                     "vision": list(CFG.VISION_ENCODERS),
                     "text": list(CFG.TEXT_ENCODERS)},
        "file_sha256": files,
    }


def register(overwrite=False):
    if os.path.exists(CFG.PREREG_PATH) and not overwrite:
        with open(CFG.PREREG_PATH) as fh:
            old = json.load(fh)
        print(f"already registered at {old['registered_utc']} -- not overwriting.")
        return old
    snap = snapshot()
    snap["registered_utc"] = _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds")
    blob = json.dumps({k: snap[k] for k in ("params", "datasets", "encoders", "file_sha256")},
                      sort_keys=True).encode()
    snap["registration_hash"] = hashlib.sha256(blob).hexdigest()
    os.makedirs(os.path.dirname(CFG.PREREG_PATH), exist_ok=True)
    with open(CFG.PREREG_PATH, "w") as fh:
        json.dump(snap, fh, indent=1)
    print(f"registered: {CFG.PREREG_PATH}")
    print(f"registration hash: {snap['registration_hash']}")
    return snap


def check(deviation_note: str | None = None):
    if not os.path.exists(CFG.PREREG_PATH):
        raise RuntimeError("no prereg.json -- run notebook 00 (register) first")
    with open(CFG.PREREG_PATH) as fh:
        reg = json.load(fh)
    now = snapshot()
    diffs = []
    for f, h in reg["file_sha256"].items():
        if now["file_sha256"].get(f) != h:
            diffs.append(f"file changed: {f}")
    if json.dumps(reg["params"], sort_keys=True) != json.dumps(now["params"], sort_keys=True):
        diffs.append("PREREG parameters changed")
    for k in ("datasets", "encoders"):
        if reg[k] != now[k]:
            diffs.append(f"{k} list changed")
    status = {"registered_utc": reg["registered_utc"],
              "registration_hash": reg["registration_hash"],
              "deviations": diffs, "deviation_note": deviation_note}
    if diffs and not deviation_note:
        raise RuntimeError("analysis differs from the registration:\n  - " + "\n  - ".join(diffs)
                           + "\nEither restore the registered files, or set DEVIATION_NOTE "
                             "in notebook 04 to a sentence explaining the change (it is "
                             "written into every results file).")
    print("pre-registration check:", "PASSED (no deviations)" if not diffs else
          f"{len(diffs)} DECLARED deviation(s)")
    return status

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/analyse_all.py
"""Run the pre-registered analysis over every pool and write the paper's numbers.

    results/pools/<dataset>.json      full per-pool record (run_pipeline.run_pool)
    results/pools/<dataset>_logits.npz  held-out logits, labels, fold masks
    results/battery.json              estimands A-F for each pre-registered subset
    results/summary.md                human-readable tables
    results/figures/*.png
"""

from __future__ import annotations

import hashlib
import json
import os
import time

import numpy as np

import config as CFG
import extract as X
import run_pipeline as RP

SUBSETS = {
    "all": ("pathology", "vision", "text"),
    "pathology": ("pathology",),
    "vision": ("vision",),
    "text": ("text",),
    "image": ("pathology", "vision"),
}


def _manifest(ds):
    with open(os.path.join(CFG.MANIFEST_DIR, f"{ds}.json")) as fh:
        return json.load(fh)


def load_pool(ds, domain):
    man = _manifest(ds)
    encs = CFG.DOMAINS[domain][1]
    feats, missing = {}, []
    for e in encs:
        if X.is_done(ds, e, man):
            feats[e] = np.load(X._paths(ds, e)[1]).astype(np.float64)
        else:
            missing.append(e)
    groups = None
    if CFG.PREREG["grouped_folds_where_groups_exist"] and man.get("groups"):
        groups = np.asarray(man["groups"])
    return feats, np.asarray(man["labels"]), groups, man, missing


def _feature_key(ds, feats, man, prereg_hash):
    h = hashlib.sha256()
    h.update(man["hash"].encode())
    h.update(prereg_hash.encode())
    for e in sorted(feats):
        with open(X._paths(ds, e)[2]) as fh:
            h.update(json.load(fh).get("saved", e).encode())
        h.update(e.encode())
    return h.hexdigest()[:16]


def run_all(prereg_status, domains=("pathology", "vision", "text"), force=False):
    P = CFG.PREREG
    out_dir = os.path.join(CFG.RESULTS_DIR, "pools")
    os.makedirs(out_dir, exist_ok=True)
    records, skipped = [], []
    for domain in domains:
        for ds in CFG.DOMAINS[domain][0]:
            jp = os.path.join(CFG.MANIFEST_DIR, f"{ds}.json")
            if not os.path.exists(jp):
                skipped.append({"dataset": ds, "reason": "no manifest (not extracted)"})
                continue
            feats, y, groups, man, missing = load_pool(ds, domain)
            if len(feats) < P["min_encoders_per_pool"]:
                skipped.append({"dataset": ds, "reason": f"only {len(feats)} encoders "
                                f"(< {P['min_encoders_per_pool']}); missing {missing}"})
                continue
            key = _feature_key(ds, feats, man, prereg_status["registration_hash"])
            rp = os.path.join(out_dir, f"{ds}.json")
            if os.path.exists(rp) and not force:
                with open(rp) as fh:
                    rec = json.load(fh)
                if rec.get("feature_key") == key:
                    print(f"[{ds}] cached result ({len(feats)} encoders)")
                    records.append(rec)
                    continue
            print(f"\n[{ds}] {domain}: {len(feats)} encoders, n={len(y)}, "
                  f"{'grouped' if groups is not None else 'ungrouped'} folds"
                  + (f"; missing {missing}" if missing else ""), flush=True)
            t0 = time.time()
            rec = RP.run_pool(ds, feats, y, groups=groups, k=P["k_folds"], seed=P["seed"],
                              n_cap_cka=P["n_cap_cka"], comp_tol=P["comp_tol_pp"],
                              cka_min=P["cka_min_gap"], cka_sd_floor=P["cka_sd_floor"],
                              fusion_rules=tuple([P["primary_fusion_rule"]] +
                                                 P["secondary_fusion_rules"]),
                              probe_backend=P["probe_backend"], return_logits=True)
            np.savez_compressed(os.path.join(out_dir, f"{ds}_logits.npz"),
                                y=rec.pop("_y"), masks=rec.pop("_masks"),
                                **{f"Z_{k}": v for k, v in rec.pop("_logits").items()})
            rec.update(domain=domain, feature_key=key, missing_encoders=missing,
                       manifest_hash=man["hash"], prereg=prereg_status)
            RP.dump(rec, rp)
            print(f"[{ds}] done in {(time.time() - t0) / 60:.1f} min: anchor={rec['anchor']} "
                  f"base={rec['base']} matched pairs={len(rec['matched_pairs'])}", flush=True)
            with open(rp) as fh:                       # use exactly what was saved
                records.append(json.load(fh))
    return records, skipped


def battery(records, prereg_status):
    out = {"prereg": prereg_status, "generated": time.strftime("%Y-%m-%d %H:%M:%S")}
    for name, doms in SUBSETS.items():
        recs = [r for r in records if r["domain"] in doms]
        if len(recs) < 2:
            out[name] = {"error": f"only {len(recs)} pools"}
            continue
        try:
            res = RP.analyse_pools(recs, margins=tuple(CFG.PREREG["equivalence_margins_pp"]))
        except Exception as e:
            res = {"error": f"{type(e).__name__}: {e}"}
        res["pools"] = [r["pool"] for r in recs]
        out[name] = res
    return out


def _f(v, nd=2):
    try:
        return f"{float(v):+.{nd}f}"
    except Exception:
        return "n/a"


def _p(v):
    try:
        v = float(v)
        return f"{v:.3f}" if v >= 0.001 else f"{v:.1e}"
    except Exception:
        return "n/a"


def summary_md(records, bat, skipped):
    L = ["# v3 results summary", "",
         f"Pre-registration: {bat['prereg']['registered_utc']} "
         f"(hash {bat['prereg']['registration_hash'][:12]}); deviations: "
         f"{bat['prereg']['deviations'] or 'none'}", ""]
    L += ["## Pools", "", "| pool | domain | n | encoders | grouped | anchor | base | matched pairs | primary D (pp) |",
          "|---|---|---|---|---|---|---|---|---|"]
    for r in records:
        prim = [p for p in r["matched_pairs"] if p.get("primary")]
        d = _f(prim[0]["diff"]) if prim else "-"
        L.append(f"| {r['pool']} | {r['domain']} | {r['n_inputs']} | {r['n_members']} | "
                 f"{r['grouped_folds']} | {r['anchor']} | {r['base'][1]} | "
                 f"{len(r['matched_pairs'])} | {d} |")
    if skipped:
        L += ["", "Skipped: " + "; ".join(f"{s['dataset']} ({s['reason']})" for s in skipped)]
    for name in SUBSETS:
        b = bat.get(name, {})
        L += ["", f"## Subset: {name}", ""]
        if "error" in b:
            L.append(f"_{b['error']}_")
            continue
        ne = [k for k, v in b.items() if isinstance(v, dict) and "not_estimable" in v]
        if ne:
            L.append("_Not estimable in this subset: " + "; ".join(
                f"{k} ({b[k]['not_estimable']})" for k in ne) + "_")
            L.append("")
        a = (b.get("A_matched_null_primary_only") or {}).get("cluster_robust", {})
        aa = (b.get("A_matched_null_all_pairs") or {}).get("cluster_robust", {})
        L.append(f"- **A (primary, one pair per pool)**: D = {_f(a.get('estimate_pp'))} pp, "
                 f"95% CI [{_f(a.get('ci_low'))}, {_f(a.get('ci_high'))}], p = {_p(a.get('p'))}, "
                 f"G = {a.get('G')}")
        L.append(f"- A (all matched pairs, clustered): D = {_f(aa.get('estimate_pp'))} pp, "
                 f"95% CI [{_f(aa.get('ci_low'))}, {_f(aa.get('ci_high'))}], "
                 f"pairs = {aa.get('n')}, G = {aa.get('G')}")
        B = b.get("B_dose_response") or {}
        B = B if "slope_pp_per_unit_cka" in B else {}
        L.append(f"- B (dose-response): slope {_f(B.get('slope_pp_per_unit_cka'))} pp/unit CKA, "
                 f"SE {_f(B.get('se'))}, p = {_p(B.get('p'))}, MDE80 = {_f(B.get('mde_80pct_power'))}; "
                 f"interpretable: {B.get('mde_80pct_power', 99) <= 0.5 if B else 'n/a'}")
        C = b.get("C_equivalence") or {}
        L.append(f"- C (TOST, all matched pairs): largest margin cleared = "
                 f"{C.get('largest_margin_cleared')} pp (G = {C.get('G')})")
        D = (b.get("D_selection_regret") or {}).get("per_rule", {})
        if D:
            L += ["", "| criterion | mean regret (pp) | median | p90 | top-1 agreement | pools |",
                  "|---|---|---|---|---|---|"]
            for k, v in D.items():
                L.append(f"| {k} | {v['mean_regret_pp']:.2f} | {v['median_regret_pp']:.2f} | "
                         f"{v['p90_regret_pp']:.2f} | {v['top1_agreement_with_oracle']:.0%} | {v['n_pools']} |")
        Em = b.get("E_mechanism_on_real") or {}
        if "joint_model" in Em:
            j = Em["joint_model"]
            L.append("")
            L.append(f"- E (joint model, cluster-robust): CKA {_f(j['cka']['coef_pp_per_unit'])} "
                     f"(p = {_p(j['cka']['p'])}), disagreement {_f(j['disagreement']['coef_pp_per_unit'])} "
                     f"(p = {_p(j['disagreement']['p'])}), accuracy {_f(j['indiv_acc']['coef_pp_per_unit'], 3)} "
                     f"(p = {_p(j['indiv_acc']['p'])}); candidates = {Em['n_candidates']}, G = {Em['G']}")
    return "\n".join(L) + "\n"


def figures(records, fig_dir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(fig_dir, exist_ok=True)
    colors = {"pathology": "#c0392b", "vision": "#2471a3", "text": "#239b56"}
    # 1. dose-response
    fig, ax = plt.subplots(figsize=(6, 4))
    for r in records:
        xs = [p["cka_gap"] for p in r["matched_pairs"]]
        ys = [p["diff"] for p in r["matched_pairs"]]
        ax.scatter(xs, ys, s=10, alpha=0.5, color=colors[r["domain"]])
    for d, c in colors.items():
        ax.scatter([], [], color=c, label=d)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("CKA gap (higher-CKA minus lower-CKA candidate)")
    ax.set_ylabel("utility difference D (pp)")
    ax.legend(frameon=False)
    ax.set_title("Matched pairs: does D grow with the CKA gap?")
    fig.tight_layout()
    fig.savefig(os.path.join(fig_dir, "dose_response.png"), dpi=200)
    plt.close(fig)
    # 2. candidate utility vs CKA and vs competence (within-pool centred)
    fig, axs = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    for r in records:
        c = r["candidates"]
        u = np.array([x["utility"] for x in c])
        k = np.array([x["cka"] for x in c])
        a = np.array([x["indiv_acc"] for x in c])
        axs[0].scatter(k, u - u.mean(), s=12, alpha=0.6, color=colors[r["domain"]])
        axs[1].scatter(a - a.mean(), u - u.mean(), s=12, alpha=0.6, color=colors[r["domain"]])
    axs[0].set_xlabel("CKA with anchor")
    axs[1].set_xlabel("individual accuracy (pp, centred within pool)")
    axs[0].set_ylabel("marginal utility (pp, centred within pool)")
    for ax in axs:
        ax.axhline(0, color="k", lw=0.8)
    fig.tight_layout()
    fig.savefig(os.path.join(fig_dir, "utility_vs_cka_and_competence.png"), dpi=200)
    plt.close(fig)


def write_all(records, bat, skipped):
    os.makedirs(CFG.RESULTS_DIR, exist_ok=True)
    RP.dump(bat, os.path.join(CFG.RESULTS_DIR, "battery.json"))
    md = summary_md(records, bat, skipped)
    with open(os.path.join(CFG.RESULTS_DIR, "summary.md"), "w") as fh:
        fh.write(md)
    figures(records, os.path.join(CFG.RESULTS_DIR, "figures"))
    return md


def verify_determinism(records, bat):
    """Recompute the battery from the saved pool JSONs and assert it matches."""
    again = battery(records, bat["prereg"])
    a = json.dumps({k: v for k, v in bat.items() if k not in ("generated",)},
                   sort_keys=True, default=float)
    b = json.dumps({k: v for k, v in again.items() if k not in ("generated",)},
                   sort_keys=True, default=float)
    ok = a == b
    print("battery recomputation from saved pool records:", "IDENTICAL" if ok else "DIFFERS")
    return ok

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/tests/test_core.py
"""Unit tests for the analysis core. Run: python tests/test_core.py"""

import os
import sys

import numpy as np

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "code"))

import folds as F          # noqa: E402
import fusion              # noqa: E402
import inference as I      # noqa: E402
import pairs as P          # noqa: E402


def test_fix_base_uses_cka_median_not_alphabetical():
    # odd number of non-anchor members: alphabetical median would be "c"
    cka = {"anchor": 0.0, "a": 0.9, "b": 0.1, "c": 0.8, "d": 0.2, "e": 0.5}
    base = P.fix_base("anchor", cka)
    assert base == ("anchor", "e"), base            # CKA order b,d,e,c,a -> median e
    cka2 = {"anchor": 0.0, "a": 0.9, "b": 0.1, "c": 0.8, "d": 0.2}
    assert P.fix_base("anchor", cka2) == ("anchor", "d")   # even: lower median


def test_torch_probe_reaches_sklearn_optimum():
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    rng = np.random.default_rng(0)
    for C in (7, 2):
        n, d = 2000, 128
        y = rng.integers(0, C, n)
        X = rng.normal(size=(C, d))[y] * 0.2 + rng.normal(size=(n, d))
        m = F.make_folds(y, k=5)
        Zt = F.per_fold_logits(X, y, m, backend="torch")
        # tightly converged sklearn reference on the first fold
        sc = StandardScaler().fit(X[~m[0]])
        ref = LogisticRegression(C=1.0, max_iter=20000, tol=1e-10).fit(sc.transform(X[~m[0]]), y[~m[0]])
        zr = ref.decision_function(sc.transform(X[m[0]]))
        if zr.ndim == 1:
            zr = np.stack([-zr / 2, zr / 2], 1)
        agree = (zr.argmax(1) == Zt[m[0]].argmax(1)).mean()
        assert agree > 0.995, (C, agree)
        assert np.corrcoef(zr.ravel(), Zt[m[0]].ravel())[0, 1] > 0.999


def test_fusion_identifiability_cases():
    """The corrected claim: no global scaling symmetry; non-identified when C < K
    or at the boundary where a useless teacher's tau -> inf."""
    rng = np.random.default_rng(1)
    m = fusion.ClassConditionalFusion(lam=0.0)
    n, K, C = 200, 3, 5
    Zs = [rng.normal(size=(n, C)) for _ in range(K)]
    theta = rng.normal(size=C * K + K + C) * 0.3
    base = m._fused(theta, Zs, K, C)
    t2 = theta.copy()
    t2[C * K:C * K + K] += np.log(2.0)            # rescale tau alone
    assert np.abs(m._fused(t2, Zs, K, C) - base).max() > 1e-3, "tau rescaling must change the fit"
    # C < K: a direction in (alpha, tau) space leaves v = alpha/tau unchanged
    C2 = 2
    W = rng.normal(size=(C2, K)); tau = np.exp(rng.normal(size=K))
    alpha = np.exp(W) / np.exp(W).sum(1, keepdims=True)
    v = alpha / tau
    # find tau' != tau with sum_k v_ck tau'_k = 1 for both classes (2 equations, 3 unknowns)
    A = v                                        # (C2, K)
    null = np.linalg.svd(A)[2][-1]               # null-space direction of A
    tau_p = tau + 0.1 * null * np.abs(tau).min() / np.abs(null).max()
    assert np.all(tau_p > 0)
    alpha_p = v * tau_p
    assert np.allclose(alpha_p.sum(1), 1.0, atol=1e-9)
    assert np.allclose(alpha_p / tau_p, v) and not np.allclose(tau_p, tau)


def test_identified_fusion_param_count():
    rng = np.random.default_rng(2)
    n, K, C = 300, 3, 7
    y = rng.integers(0, C, n)
    Zs = [np.eye(C)[y] * 2 + rng.normal(size=(n, C)) for _ in range(K)]
    f = fusion.IdentifiedClassConditionalFusion().fit(Zs, y)
    assert f.n_params == C * K + C == 28


def test_mixed_effects_close_to_cluster_robust():
    rng = np.random.default_rng(3)
    cl = np.repeat(np.arange(15), 6)
    y = 0.2 + np.repeat(rng.normal(0, 0.3, 15), 6) + rng.normal(0, 1, 90)
    me = I.mixed_effects_random_intercept(y, cl)
    cr = I.cluster_robust_ols(y, cl)
    assert abs(me["estimate"] - cr["estimate"][0]) < 0.05
    assert 0.5 < me["se"] / cr["se"][0] < 2.0, (me["se"], cr["se"][0])


if __name__ == "__main__":
    for name, fn in sorted(globals().items()):
        if name.startswith("test_") and callable(fn):
            fn()
            print(f"PASS  {name}")
    print("\nall core checks passed")

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/tests/test_analysis_e2e.py
"""End-to-end analysis test on synthetic pools (CPU, ~2-4 min).

Writes synthetic features + manifests exactly where notebooks 01-03 would, then
runs the real notebook-04 path: prereg.register -> prereg.check -> run_all ->
battery -> write_all -> verify_determinism. Also checks that the prereg guard
blocks an undeclared change and accepts a declared one.
"""

import json
import os
import shutil
import sys

import numpy as np

ROOT = "/tmp/v3_analysis"
shutil.rmtree(ROOT, ignore_errors=True)
os.environ["V3_DRIVE"] = os.path.join(ROOT, "MyDrive")
os.environ["V3_LOCAL"] = os.path.join(ROOT, "local")
HERE = os.path.dirname(os.path.abspath(__file__))
SRC = os.path.join(HERE, "..", "code")
sys.path.insert(0, SRC)

import config as CFG  # noqa: E402

os.makedirs(CFG.CODE_DIR, exist_ok=True)
for f in os.listdir(SRC):
    if f.endswith(".py"):
        shutil.copy(os.path.join(SRC, f), CFG.CODE_DIR)

import imgcache as IC    # noqa: E402
import synth_pool as SP  # noqa: E402

# two synthetic "domains", 4 pools each, 7 members per pool
MEMBERS = [f"synth_{i:02d}" for i in range(7)]
CFG.PATHOLOGY_DATASETS = {f"sp{i}": dict(cap=None, persist=False) for i in range(4)}
CFG.VISION_DATASETS = {f"sv{i}": dict(cap=None, persist=False) for i in range(4)}
CFG.TEXT_DATASETS = {}
enc = {m: dict(kind="timm", id="synthetic") for m in MEMBERS}
CFG.PATHOLOGY_ENCODERS = enc
CFG.VISION_ENCODERS = enc
CFG.TEXT_ENCODERS = {}
CFG.DOMAINS = {"pathology": (CFG.PATHOLOGY_DATASETS, enc), "vision": (CFG.VISION_DATASETS, enc),
               "text": ({}, {})}
CFG.PREREG["n_cap_cka"] = 1500

import extract as X  # noqa: E402

for j, ds in enumerate(list(CFG.PATHOLOGY_DATASETS) + list(CFG.VISION_DATASETS)):
    pool = SP.make_synthetic_pool(n_members=7, n_classes=4, n_per_class=150, d_total=64,
                                  seed=100 + j)
    y = pool.y
    groups = [f"g{i % 30}" for i in range(len(y))] if j % 2 == 0 else None
    man = dict(name=ds, n=len(y), labels=y.tolist(), uids=[f"{ds}_{i}" for i in range(len(y))],
               groups=groups, classes=[str(c) for c in range(4)])
    man["hash"] = IC.manifest_hash(man["uids"], man["labels"])
    os.makedirs(CFG.MANIFEST_DIR, exist_ok=True)
    json.dump(man, open(os.path.join(CFG.MANIFEST_DIR, f"{ds}.json"), "w"))
    for m in MEMBERS:
        d, npy, meta, _ = X._paths(ds, m)
        os.makedirs(d, exist_ok=True)
        np.save(npy, pool.X[m].astype(np.float32))
        json.dump({"manifest_hash": man["hash"], "complete": True, "saved": "t"}, open(meta, "w"))

import analyse_all as A  # noqa: E402
import prereg            # noqa: E402

reg = prereg.register()
status = prereg.check()
assert status["deviations"] == []

# guard: undeclared change must be refused, declared change accepted
p = os.path.join(CFG.CODE_DIR, "pairs.py")
orig = open(p).read()
open(p, "a").write("\n# edit\n")
try:
    prereg.check()
    raise AssertionError("prereg guard failed to block an undeclared change")
except RuntimeError:
    pass
st2 = prereg.check(deviation_note="test deviation")
assert st2["deviations"] == ["file changed: pairs.py"]
open(p, "w").write(orig)
print("PASS  prereg: registers, blocks undeclared change, records declared deviation")

CFG.PREREG["probe_backend"] = "torch"
records, skipped = A.run_all(status, domains=("pathology", "vision"))
assert len(records) == 8 and not skipped, skipped
assert any(r["grouped_folds"] for r in records) and not all(r["grouped_folds"] for r in records)
bat = A.battery(records, status)
for sub in ("all", "pathology", "vision", "image"):
    assert "error" not in bat[sub], (sub, bat[sub])
    assert bat[sub]["D_selection_regret"]["n_pools_scored"] > 0

# Regression test for the Colab failure: a subset in which only ONE pool has
# matched pairs must report A as not estimable, not crash the whole subset.
import run_pipeline as RP  # noqa: E402
one = [dict(r) for r in records]
keep = next(i for i, r in enumerate(one) if r["matched_pairs"])
for i, r in enumerate(one):
    if i != keep:
        r["matched_pairs"] = []
res = RP.analyse_pools(one)
assert "not_estimable" in res["A_matched_null_primary_only"], res["A_matched_null_primary_only"]
assert res["D_selection_regret"]["n_pools_scored"] == len(one)
none = [dict(r, matched_pairs=[]) for r in records]
res0 = RP.analyse_pools(none)
assert "not_estimable" in res0["A_matched_null_all_pairs"]
md_one = A.summary_md(one, {"prereg": status, "all": res}, [])
assert "Not estimable" in md_one
print("PASS  estimands fail independently (1 pool / 0 pools with matched pairs)")
md = A.write_all(records, bat, skipped)
assert os.path.exists(os.path.join(CFG.RESULTS_DIR, "figures", "dose_response.png"))
assert A.verify_determinism(records, bat)
print(md[:1800])

# cached re-run must reuse every pool
records2, _ = A.run_all(status, domains=("pathology", "vision"))
assert [r["pool"] for r in records2] == [r["pool"] for r in records]
print("PASS  analysis: 8 pools, all subsets, summary, figures, determinism, cache reuse")
print("\nall analysis checks passed")

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/tests/test_resume_after_reset.py
"""Regression tests for the two failures seen on Colab after a runtime reset.

1. A saved manifest whose downloaded source images are gone (runtime reset)
   must trigger a re-download, verified against the saved manifest hash,
   instead of a FileNotFoundError inside a DataLoader worker.
2. A rebuilt dataset that does NOT reproduce the saved row order is refused.
3. Hibou-B's remote code imports `transformers.onnx` (removed in transformers 5);
   the stand-in module must make that import work.
"""
import os, shutil, sys
import numpy as np
ROOT = "/tmp/v3_reset"
shutil.rmtree(ROOT, ignore_errors=True)
os.environ["V3_DRIVE"] = os.path.join(ROOT, "MyDrive")
os.environ["V3_LOCAL"] = os.path.join(ROOT, "local")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "code"))
from PIL import Image
import config as CFG
import imgcache as IC

src_root = os.path.join(CFG.LOCAL_DL, "CRC-VAL-HE-7K")
calls = {"n": 0}

def fake_download_and_tree(shuffle=False):
    calls["n"] += 1
    rng = np.random.default_rng(0)
    for c in ("ADI", "BACK", "DEB"):
        for i in range(12):
            p = os.path.join(src_root, c, f"{c}-TCGA-{i:04d}.tif")
            os.makedirs(os.path.dirname(p), exist_ok=True)
            if not os.path.exists(p):
                Image.fromarray((rng.random((64, 64, 3)) * 255).astype(np.uint8)).save(p)
    classes, files, labels = IC._folder_tree(src_root)
    if shuffle:
        files, labels = files[::-1], labels[::-1]
    return dict(uids=[os.path.relpath(f, src_root) for f in files], labels=labels, groups=None,
                classes=classes, reader=("files", files))

IC.BUILDERS["crc_val_7k"] = fake_download_and_tree
arr, man = IC.ensure_images("crc_val_7k", workers=2, chunk=16)
ref = np.asarray(arr).copy()
assert calls["n"] == 1
# --- runtime reset with the decoded cache on Drive: restored, NO re-download
assert os.path.exists(os.path.join(CFG.IMG_DRIVE_DIR, "crc_val_7k.npy")), "downloaded sets persist"
shutil.rmtree(CFG.LOCAL)
arr1, _ = IC.ensure_images("crc_val_7k", workers=2, chunk=16)
assert calls["n"] == 1 and np.array_equal(np.asarray(arr1), ref)
print("PASS  runtime reset: decoded cache restored from Drive, nothing re-downloaded")
# --- runtime reset WITHOUT a Drive cache: sources re-downloaded and verified
shutil.rmtree(CFG.LOCAL)
for f in os.listdir(CFG.IMG_DRIVE_DIR):
    os.remove(os.path.join(CFG.IMG_DRIVE_DIR, f))
arr2, man2 = IC.ensure_images("crc_val_7k", workers=2, chunk=16)
assert calls["n"] == 2, "sources should have been re-downloaded"
assert man2["hash"] == man["hash"] and np.array_equal(np.asarray(arr2), ref)
print("PASS  runtime reset without Drive cache: re-downloaded, manifest hash verified, identical images")

# --- a rebuild that changes row order must be refused
shutil.rmtree(CFG.LOCAL)
for f in os.listdir(CFG.IMG_DRIVE_DIR):
    os.remove(os.path.join(CFG.IMG_DRIVE_DIR, f))
IC.BUILDERS["crc_val_7k"] = lambda: fake_download_and_tree(shuffle=True)
try:
    IC.ensure_images("crc_val_7k", workers=0)
    raise AssertionError("row-order change was not detected")
except RuntimeError as e:
    assert "does not reproduce the saved manifest" in str(e)
print("PASS  changed row order after re-download is refused")

# --- transformers.onnx stand-in
sys.modules["transformers.onnx"] = None          # make the real import fail, as on transformers 5
import encoders as E
E._shim_transformers_onnx()
from transformers.onnx import OnnxConfig
class Dinov2OnnxConfig(OnnxConfig):               # what Hibou's configuration file does
    pass
print("PASS  transformers.onnx stand-in lets Hibou-style remote config import")
print("\nall reset checks passed")

## 2. Check your data and checkpoint paths

In [ ]:
import sys, importlib
sys.path.insert(0, f"{PROJECT}/code")
import config as CFG; importlib.reload(CFG)

def count(d, exts=(".png", ".jpg", ".jpeg", ".tif", ".tiff")):
    try:
        return sum(1 for f in os.listdir(d) if f.lower().endswith(exts))
    except FileNotFoundError:
        return None

checks = []
for split in ("train", "val"):
    p = os.path.join(CFG.BRACS_DIR, split)
    n = sum(count(os.path.join(p, c)) or 0 for c in (os.listdir(p) if os.path.isdir(p) else []))
    checks.append((f"BRACS {split}", p, n, {"train": 3657, "val": 312}[split]))
checks.append(("SICAPv2 images", os.path.join(CFG.SICAP_DIR, "images"),
               count(os.path.join(CFG.SICAP_DIR, "images")), 18783))
for x in ("Train.xlsx", "Test.xlsx"):
    p = os.path.join(CFG.SICAP_DIR, "partition", "Test", x)
    checks.append((f"SICAPv2 {x}", p, int(os.path.exists(p)), 1))
checks.append(("MHIST images", os.path.join(CFG.MHIST_DIR, "images"),
               count(os.path.join(CFG.MHIST_DIR, "images")), 3152))
csvs = [c for c in CFG.MHIST_CSV_CANDIDATES if os.path.exists(os.path.join(CFG.MHIST_DIR, c))]
checks.append(("MHIST annotations", CFG.MHIST_DIR, len(csvs), 1))
checks.append(("CTransPath checkpoint", CFG.CTRANSPATH_CKPT, int(os.path.exists(CFG.CTRANSPATH_CKPT)), 1))
ok_all = True
for name, p, got, want in checks:
    ok = got == want
    ok_all &= ok
    print(f"{'OK ' if ok else 'XX '} {name:24s} found={got}  expected={want}   {p}")
print("\nv2 caches (only used for the consistency check in notebook 01):")
for ds, d in CFG.V2_CACHE.items():
    print(f"   {ds:8s} {'present' if os.path.isdir(d) else 'absent'}   {d}")
print("\n" + ("ALL DATA PATHS OK" if ok_all else
       "*** Fix the paths marked XX in config.py (PATHS block) and re-run the config.py cell ***"))

## 3. Hugging Face access for gated encoders
Downloads one small config file per gated repo. "no access" means the licence is not accepted for your account yet.

In [ ]:
import encoders as E; importlib.reload(E)
have_token = E.hf_login()
from huggingface_hub import hf_hub_download
for name, spec in CFG.PATHOLOGY_ENCODERS.items():
    if not spec.get("gated"):
        continue
    repo = spec["id"].split(":", 1)[-1]
    try:
        hf_hub_download(repo, "config.json")
        print(f"OK          {name:14s} {repo}")
    except Exception as e:
        print(f"no access   {name:14s} {repo}   ({type(e).__name__})")

## 4. Test suite (hard gate)
Core unit tests, then the full analysis path on synthetic pools (~3–5 min on CPU).

In [ ]:
import subprocess
for t in ("test_core.py", "test_analysis_e2e.py", "test_resume_after_reset.py"):
    r = subprocess.run([sys.executable, f"{PROJECT}/tests/{t}"], capture_output=True, text=True)
    lines = [l for l in r.stdout.splitlines() if l.startswith(("PASS", "all ", "FAIL"))]
    print("\n".join(lines))
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit(f"{t} FAILED - do not continue")
print("\nALL TESTS PASSED")

## 5. Register the analysis
Freezes the analysis parameters and code hashes in `TMLR_v3/prereg.json`.

**Then upload `prereg.json` to OSF (osf.io → new project → upload; or "Registrations")** or commit it to a
GitHub repo and tag the commit. A third-party timestamp is what lets reviewers check that the analysis was fixed
before you saw any real-pool result. Do this **before running notebook 04**.

In [ ]:
import json, prereg; importlib.reload(prereg)
reg = prereg.register()
print(json.dumps(reg["params"], indent=1))

In [ ]:
print(json.dumps({k: reg[k] for k in ("registered_utc", "registration_hash")}, indent=1))
print("\nNext: notebook 01_pathology_features.ipynb on a GPU runtime.")